# Predicting Credit Card Fraud — LDS1071 Final Project

**Kaggle:** [Playground Series S6E11](https://www.kaggle.com/competitions/playground-series-s6e11)  
**Task:** estimate $p_i=\Pr(Y_i=1\mid X_i)$; Kaggle ranks submissions using **ROC AUC**.  
**Approach:** exploratory data analysis → fold-safe preprocessing → SVD/PCA experiment → logistic regression, random forest, XGBoost → spline/nonlinear logistic regression → five-fold stratified CV → offset boosting over logistic log-odds → retain *all candidate* out-of-fold (OOF) predictions → probability ensembles → submission.

### Terminology

- **OOF predictions:** each training row's score comes from a model fitted *without* that row.
- **SVD/PCA:** $X_c=U\Sigma V^\top$; PCA projects data onto dominant right singular vectors after appropriate preprocessing/centering. Retaining 90% of *feature variance* does **not** guarantee retaining 90% of fraud-predictive signal.
- **Bagging:** fit base learners on bootstrap samples, then aggregate. This project instead uses **cross-validation ensembling** (five fitted folds per candidate) and **heterogeneous model blending** (across different model families). Random forest itself uses bagging-like tree sampling.

### Fair evaluation

Use identical shuffled `StratifiedKFold(n_splits=5)` splits for every candidate. Fit all imputation/encoding/scaling/PCA **inside each fold**. Select candidates and blends using OOF scores only; never touch Kaggle test labels. **Caveat:** when hyperparameters and blends are selected on these same OOF scores, the *selected* OOF score is optimistic; nested CV or a separate holdout would be required for an unbiased final performance estimate. Do not use public-leaderboard feedback as a substitute.

**Reproducibility:** `RANDOM_STATE=42`; results, split definitions, fold metrics, all OOF/test predictions, and fitted fold models are saved to `outputs/` by default (ignored by Git). No raw competition data is bundled with the notebook.

**Extended visualization audit:** seaborn class imbalance, distributions, Pearson/Spearman dependencies, categorical/flag fraud rates, quantile-binned nonlinear profiles, interaction heatmaps, robust univariate tails, and an unsupervised Isolation Forest anomaly score. Figure inputs are explicitly identified as full-data aggregates or fixed-seed visualization samples. Diagnostic analyses do **not** automatically remove unusual transactions.


**Logistic-model assumptions are audited explicitly in Sections 5.1 and 8.3**, with saved aggregate diagnostic tables and plots.


**Expanded visual diagnostics (experiment004).** The notebook additionally visualizes (i) single-feature discrimination; (ii) class-conditional empirical CDFs; (iii) categorical × flag interactions; (iv) hour × weekday fraud rates; (v) representative pairwise feature geometry; (vi) feature-drift checks; (vii) OOF score distributions, capture/lift, precision-at-budget, decision errors, calibration, subgroup breakdowns, model-disagreement, efficiency, and feature importance/partial dependence; and (viii) ensemble diagnostic comparisons. Plots are descriptive, not independent test-set evaluations, and they do not alter model selection. Individual model plots use predictions from held-out CV rows, not in-sample predictions.


## 1. Environment and reproducibility

Set `FAST_DEV_RUN=True` for a limited smoke test before the full dataset. The final deliverable should use `False` and `MAX_CANDIDATES_PER_FAMILY=None`. Parallelize **within** the random forest or XGBoost fit (`n_jobs=-1`), not simultaneously at the outer CV/search level, to avoid CPU oversubscription.

The EDA plots use a fixed-seed sample for point-level graphics and **all training rows for rates and counts**. Figures are saved to ignored `outputs/figures/`. `seaborn` provides consistent professional styling.


In [ ]:
from __future__ import annotations

import json
import os
import time
import zipfile
import warnings
from pathlib import Path

import joblib
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    average_precision_score, brier_score_loss, f1_score,
    log_loss, precision_score, recall_score, roc_auc_score,
    roc_curve, precision_recall_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.inspection import partial_dependence
from scipy.stats import ks_2samp
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from xgboost import XGBClassifier
import xgboost

RANDOM_STATE = 42
N_SPLITS = 5
N_JOBS = -1  # Internal parallelism for forests and XGBoost; CV loop is sequential.
FAST_DEV_RUN = False
DEV_SAMPLE_SIZE = 15_000
MAX_CANDIDATES_PER_FAMILY = None  # Set to 1 for a very short diagnostic run.
EDA_SAMPLE_SIZE = 30_000  # Point-level figures only; full-data group rates remain unchanged.
ANOMALY_SAMPLE_SIZE = 30_000  # Limited-cost unsupervised anomaly demonstration.
ASSUMPTION_SAMPLE_SIZE = 40_000  # Design-matrix diagnostics, not an assumption proof.
SHIFT_SAMPLE_SIZE = 15_000  # Each dataset, for train/test feature-shift screening.
RUN_PDP = True  # One small-sample partial-dependence diagnostic per selected model.
SAVE_FOLD_MODELS = True  # Persist every fitted fold pipeline; set False if disk is limited.

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(exist_ok=True, parents=True)
BLAS_LIMIT = threadpool_limits(limits=1, user_api='blas')  # Avoid threaded BLAS oversubscription in CV/PCA.
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 160, 'axes.titleweight': 'semibold'})

def finish_plot(fig, filename: str):
    """Display and save a readable figure without leaving open Matplotlib handles."""
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f'{filename}.png', bbox_inches='tight')
    plt.show()
    plt.close(fig)
print({'scikit-learn': sklearn_version, 'xgboost': xgboost.__version__,
       'python_workers': os.cpu_count(), 'fast_dev_run': FAST_DEV_RUN})

## 2. Load and validate the competition data

Put `train.csv`, `test.csv`, and `sample_submission.csv` under `final_project/data/`, **or** place `playground-series-s6e11.zip` beside the notebook or in `data/`. When running on Kaggle, `/kaggle/input/playground-series-s6e11/` is also detected. Notebook execution is relative to its working directory.

In [ ]:
DATA_DIRS = [Path('data'), Path('final_project/data'),
             Path('/kaggle/input/playground-series-s6e11')]
ZIP_CANDIDATES = [Path('playground-series-s6e11.zip'),
                  Path('data/playground-series-s6e11.zip'),
                  Path('../playground-series-s6e11.zip')]

def read_competition_csv(filename: str) -> pd.DataFrame:
    for folder in DATA_DIRS:
        path = folder / filename
        if path.exists():
            return pd.read_csv(path)
    for zipped in ZIP_CANDIDATES:
        if zipped.exists():
            with zipfile.ZipFile(zipped) as archive:
                with archive.open(filename) as stream:
                    return pd.read_csv(stream)
    raise FileNotFoundError(
        f'Cannot locate {filename}. Put the extracted CSVs in data/ or the original ZIP next to the notebook.'
    )

train_full = read_competition_csv('train.csv')
test = read_competition_csv('test.csv')
sample_submission = read_competition_csv('sample_submission.csv')
TARGET, ID = 'is_fraud', 'id'
assert set(train_full[TARGET].dropna().unique()) <= {0, 1}
assert train_full[TARGET].notna().all()
assert train_full[ID].is_unique and test[ID].is_unique
assert TARGET not in test.columns
assert list(sample_submission.columns) == [ID, TARGET]
assert sample_submission[ID].equals(test[ID])
assert set(train_full.columns) - {TARGET} == set(test.columns)

print('Train:', train_full.shape, 'Test:', test.shape)
print('Target prevalence:', f'{train_full[TARGET].mean():.4%}')
display(train_full.head(5))

## 3. Explore the data before modeling

**Questions:** How imbalanced is the target? What are the numerical distributions and missingness? Which variables appear associated with fraud, including monotone and nonmonotone patterns? Are there unusual but valid transactions?

Plots of **rates/counts** use all training rows; dense point-level plots use a documented fixed-seed sample (`EDA_SAMPLE_SIZE`). These are exploratory **associations**, not causal effects. `id` is an identifier, not a predictor. Avoid leaking an exploratory preprocessing transformation into CV: trained imputation, encoding, scaling, and spline/PCA transformations occur within the pipeline. The Kaggle test set is not used to design models or data-dependent features.


In [ ]:
categorical_cols = train_full.drop(columns=[TARGET, ID]).select_dtypes(include=['object', 'category']).columns.tolist()
boolean_cols = train_full.drop(columns=[TARGET, ID]).select_dtypes(include=['bool']).columns.tolist()
continuous_cols = train_full.drop(columns=[TARGET, ID]).select_dtypes(include=['number']).columns.tolist()
print(f'Numerical: {len(continuous_cols)}; categorical: {len(categorical_cols)}; Boolean: {len(boolean_cols)}')
print('Categorical cardinalities:')
display(train_full[categorical_cols].nunique().rename('levels').to_frame())
display(train_full.isna().sum().rename('missing_count').to_frame().query('missing_count > 0'))

# Fixed-seed representative sample for heavy, observation-level plots. Fraud rates are NOT computed from it.
eda_sample = train_full.sample(n=min(EDA_SAMPLE_SIZE, len(train_full)), random_state=RANDOM_STATE).copy()
print('EDA plot sample:', eda_sample.shape, '| true full-data fraud rate:', f'{train_full[TARGET].mean():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
counts = train_full[TARGET].value_counts().reindex([0, 1], fill_value=0)
sns.barplot(x=['Legitimate (0)', 'Fraud (1)'], y=counts.values, ax=axes[0], palette=['#4C78A8', '#E45756'], hue=['Legitimate (0)', 'Fraud (1)'], legend=False)
for i, count in enumerate(counts):
    axes[0].annotate(f'{count:,} ({count/len(train_full):.2%})', (i, count), ha='center', va='bottom', fontsize=10)
axes[0].set(title='Full training set: class imbalance', xlabel='Observed transaction class', ylabel='Number of transactions')
axes[0].set_ylim(0, counts.max() * 1.12)
missing = train_full.drop(columns=[ID]).isna().mean().sort_values(ascending=False)
sns.barplot(x=missing.values * 100, y=missing.index, color='#4C78A8', ax=axes[1])
axes[1].set(title='Missingness by variable (all training rows)', xlabel='Missing observations (%)', ylabel='Feature or target')
axes[1].set_xlim(0, max(1, missing.max() * 110))
finish_plot(fig, '01_target_and_missingness')

### 3.1 Numerical distributions — original scales and tail behavior

Histograms answer whether a Gaussian-like assumption is plausible, whether features are skewed, and whether a handful of extreme values dominate the horizontal range. The sigmoid form of logistic regression **does not require Gaussian predictors**, but large skew and extreme values can make a linear predictor less adequate. Log-scaled x-axes help visualize strictly positive, right-skewed amounts.

In [ ]:
display_features = [c for c in ['amount_usd', 'hours_since_last_txn', 'distance_from_home_km',
                'account_balance_usd', 'velocity_score', 'merchant_risk_score'] if c in continuous_cols]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.flat, display_features):
    sns.histplot(data=eda_sample, x=feature, bins=45, color='#4C78A8', ax=ax, stat='count', edgecolor=None)
    if feature in ('amount_usd', 'account_balance_usd') and (eda_sample[feature] > 0).all():
        ax.set_xscale('log')
        ax.set_xlabel(f'{feature} (logarithmic x-axis)')
    else:
        ax.set_xlabel(feature)
    ax.set(title=f'Distribution: {feature}', ylabel='Transactions in EDA sample')
for ax in axes.flat[len(display_features):]: ax.axis('off')
fig.suptitle('Numerical feature distributions: random observation sample', fontsize=15, y=1.01)
finish_plot(fig, '02_numeric_distributions')

### 3.2 Class-conditional distributions and extreme values

The panels below show **within-class density**, not class prevalence. To make the rare fraud class visible, a fixed number of examples per class is sampled; **do not interpret the two density curves as posterior fraud probabilities**. Overlap suggests why any one variable is insufficient for fraud classification.

In [ ]:
shape_features = [c for c in ['amount_usd', 'hours_since_last_txn', 'velocity_score', 'merchant_risk_score'] if c in continuous_cols]
per_class = []
for label in [0, 1]:
    group = train_full.loc[train_full[TARGET].eq(label)]
    per_class.append(group.sample(n=min(3_500, len(group)), random_state=RANDOM_STATE))
balanced_view = pd.concat(per_class).assign(
    transaction_class=lambda frame: frame[TARGET].map({0: 'Legitimate', 1: 'Fraud'}))
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, feature in zip(axes.flat, shape_features):
    sns.histplot(data=balanced_view, x=feature, hue='transaction_class',
                 stat='density', common_norm=False, element='step', fill=False,
                 bins=45, ax=ax, palette={'Legitimate': '#4C78A8', 'Fraud': '#E45756'})
    if feature == 'amount_usd':
        ax.set_xscale('log')
        ax.set_xlabel('Transaction amount (USD; logarithmic x-axis)')
    else: ax.set_xlabel(feature)
    ax.set(title=f'{feature}: conditional densities', ylabel='Density within each class')
fig.suptitle('Class-conditional feature distributions (balanced visualization subsample)', y=1.01)
finish_plot(fig, '03_class_conditional_densities')

### 3.3 Correlation: linear versus monotone association

Pearson correlation measures **linear** association; Spearman correlation uses ranks to measure **monotone** association. A variable may have low correlation with fraud even when it has a strong *nonmonotone* relationship, especially when interactions matter. Correlation does not imply causation or conditional predictive value. These matrices are descriptive and are computed on all training observations.

In [ ]:
corr_features = continuous_cols + boolean_cols + [TARGET]
pearson_corr = train_full[corr_features].astype(float).corr(method='pearson')
spearman_corr = train_full[corr_features].astype(float).corr(method='spearman')
fig, axes = plt.subplots(1, 2, figsize=(19, 7.5))
for ax, matrix, title in zip(axes, [pearson_corr, spearman_corr],
                              ['Pearson correlations (linear)', 'Spearman correlations (rank / monotone)']):
    sns.heatmap(matrix, vmin=-1, vmax=1, center=0, cmap='vlag',
                linewidths=.25, square=True, ax=ax,
                cbar_kws={'label': 'Correlation coefficient', 'shrink': .7})
    ax.set_title(title)
    ax.tick_params(axis='x', labelrotation=65, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
fig.suptitle('All training rows; identifier excluded', y=1.01, fontsize=15)
finish_plot(fig, '04_pearson_spearman_correlations')
assoc = pd.DataFrame({'Pearson_with_fraud': pearson_corr[TARGET],
                      'Spearman_with_fraud': spearman_corr[TARGET]}).drop(index=TARGET)
display(assoc.reindex(assoc['Spearman_with_fraud'].abs().sort_values(ascending=False).index).head(15).round(3))

### 3.4 Categorical effects and uncertainty

Group-level fraud rates are calculated from **all observations**. We show Wilson 95% binomial intervals (descriptive, not causal) and category support: rare categories may have volatile percentages. The horizontal dotted line is the overall base fraud rate.

In [ ]:
def wilson_bounds(successes, totals, z=1.959963984540054):
    n = np.asarray(totals, dtype=float)
    k = np.asarray(successes, dtype=float)
    p = np.divide(k, n, out=np.zeros_like(k), where=n > 0)
    denom = 1.0 + z * z / np.maximum(n, 1)
    mid = (p + z * z / (2 * np.maximum(n, 1))) / denom
    half = z * np.sqrt((p * (1 - p) + z * z / (4 * np.maximum(n, 1))) / np.maximum(n, 1)) / denom
    return np.clip(mid-half, 0, 1), np.clip(mid+half, 0, 1)

cat_features = [c for c in ['merchant_category', 'auth_method', 'channel', 'device_type'] if c in categorical_cols]
fig, axes = plt.subplots(2, 2, figsize=(17, 12))
for ax, feature in zip(axes.flat, cat_features):
    summary = (train_full.groupby(feature, observed=True)[TARGET]
               .agg(frauds='sum', count='size').assign(rate=lambda a: a.frauds/a['count'])
               .sort_values('rate', ascending=False).head(9))
    lo, hi = wilson_bounds(summary.frauds.values, summary['count'].values)
    summary = summary.copy(); summary['label'] = [f'{item} (n={count:,})' for item, count in zip(summary.index, summary['count'])]
    sns.barplot(data=summary, y='label', x='rate', color='#4C78A8', ax=ax, order=summary['label'].tolist())
    for i, (_, row) in enumerate(summary.iterrows()):
        ax.plot([lo[i], hi[i]], [i, i], color='#253858', linewidth=1.9)
    ax.axvline(train_full[TARGET].mean(), ls='--', lw=1.3, color='#E45756', label='Overall fraud rate')
    ax.set(title=f'Fraud rate by {feature}', xlabel='Observed fraction fraudulent', ylabel='Category (sample size)')
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
    ax.legend(loc='lower right', fontsize=9)
for ax in axes.flat[len(cat_features):]: ax.axis('off')
fig.suptitle('Categorical associations with fraud (full training set; 95% Wilson intervals)', y=1.01, fontsize=15)
finish_plot(fig, '05_categorical_risks')

### 3.5 Boolean flags: prevalence versus fraud risk

A binary flag can be rare yet highly associated with fraud when present. We therefore compare both its **frequency of occurrence** and the **fraud rate among flagged transactions**; these denominators are different. Intervals quantify sampling uncertainty in each flagged subgroup.

In [ ]:
flag_rows = []
for feature in boolean_cols:
    flagged = train_full[feature].astype(bool)
    n_flag = int(flagged.sum())
    k_flag = int(train_full.loc[flagged, TARGET].sum())
    lo, hi = wilson_bounds(np.array([k_flag]), np.array([n_flag]))
    flag_rows.append({'flag': feature, 'flagged_count': n_flag, 'flag_prevalence': n_flag/len(train_full),
                      'fraud_rate_when_true': k_flag/max(n_flag, 1), 'lo':lo[0], 'hi':hi[0]})
flags = pd.DataFrame(flag_rows).sort_values('fraud_rate_when_true', ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.barplot(data=flags, y='flag', x='fraud_rate_when_true', color='#E45756', ax=axes[0])
for i, (_, row) in enumerate(flags.iterrows()): axes[0].plot([row.lo, row.hi], [i, i], color='#253858', lw=2)
axes[0].axvline(train_full[TARGET].mean(), ls='--', color='#253858', label='Overall fraud rate')
axes[0].set(title='Fraud rate when each Boolean flag is True', xlabel='Fraction fraudulent; 95% Wilson intervals', ylabel='Boolean feature')
axes[0].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1)); axes[0].legend()
sns.barplot(data=flags, y='flag', x='flag_prevalence', color='#4C78A8', ax=axes[1])
axes[1].set(title='How frequently each flag occurs', xlabel='Fraction of all transactions', ylabel='Boolean feature')
axes[1].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
fig.suptitle('Boolean indicators: distinct denominators, full training set', y=1.02)
finish_plot(fig, '06_flag_prevalence_vs_fraud')
display(flags.round(4))

### 3.6 Detect nonlinear effects with quantile-binned fraud probabilities

For each predictor, divide observations into approximately equal-frequency bins (or use discrete values when few exist). Estimate \(\hat p_j=k_j/n_j\), where \(k_j\) is the number of frauds in bin \(j\), and plot 95% Wilson intervals. A visible curved, U-shaped, or threshold-like trend can motivate **spline logistic regression** or tree ensembles. These are **marginal** relationships and may arise from confounding; they do not establish a causal or conditional effect. Rate curves use all training observations, with no fitted smoothing that could obscure count support.

In [ ]:
nonlinear_features = [c for c in ['amount_usd', 'hours_since_last_txn', 'distance_from_home_km',
                    'velocity_score', 'merchant_risk_score', 'account_balance_usd'] if c in continuous_cols]
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
nonlinear_summaries = []
for ax, feature in zip(axes.flat, nonlinear_features):
    feature_data = train_full[[feature, TARGET]].dropna().copy()
    feature_data['bin'] = pd.qcut(feature_data[feature], q=15, duplicates='drop')
    binned = (feature_data.groupby('bin', observed=True)
              .agg(x=(feature, 'median'), frauds=(TARGET, 'sum'), n=(TARGET, 'size')).reset_index(drop=True))
    binned['fraud_rate'] = binned['frauds']/binned['n']
    binned['lo'], binned['hi'] = wilson_bounds(binned['frauds'], binned['n'])
    binned['feature'] = feature; nonlinear_summaries.append(binned)
    sns.lineplot(data=binned, x='x', y='fraud_rate', marker='o', color='#4C78A8', ax=ax,
                 label='Fraud rate by quantile bin')
    ax.fill_between(binned['x'].to_numpy(), binned['lo'].to_numpy(), binned['hi'].to_numpy(),
                    alpha=.17, color='#4C78A8', label='95% Wilson interval')
    ax.axhline(train_full[TARGET].mean(), ls='--', color='#E45756', label='Overall fraud rate')
    if feature in ('amount_usd', 'account_balance_usd') and (binned['x'] > 0).all():
        ax.set_xscale('log'); ax.set_xlabel(f'{feature} (log x-axis)')
    else: ax.set_xlabel(feature)
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
    ax.set(title=f'Nonlinear diagnostic: {feature}', ylabel='Observed fraud rate')
    ax.legend(fontsize=7)
for ax in axes.flat[len(nonlinear_features):]: ax.axis('off')
fig.suptitle('Fraud rate profiles by predictor quantile (all labeled training rows)', fontsize=15, y=1.01)
finish_plot(fig, '07_nonlinear_binned_fraud_rates')
pd.concat(nonlinear_summaries, ignore_index=True).to_csv(OUTPUT_DIR / 'nonlinear_binned_rates.csv', index=False)

### 3.7 Time-of-day and day-of-week effects

Hour and weekday are *periodic*, not ordinary unbounded coordinates. For example, 23:00 and 00:00 are close in time despite being far apart as integers. A raw linear model assigns them a large numerical separation; trees may learn threshold effects, and a future experiment could encode time cyclically with sine/cosine terms.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, feature, tick_label in [(axes[0], 'time_of_day_hour', 'Hour of day (0–23)'),
                                (axes[1], 'day_of_week', 'Day-of-week code (0–6)')]:
    grp = train_full.groupby(feature)[TARGET].agg(['sum', 'size']).reset_index()
    grp['rate'] = grp['sum']/grp['size']
    grp['lo'], grp['hi'] = wilson_bounds(grp['sum'], grp['size'])
    sns.lineplot(data=grp, x=feature, y='rate', marker='o', ax=ax, color='#4C78A8', label='Group fraud rate')
    ax.fill_between(grp[feature].to_numpy(), grp['lo'].to_numpy(), grp['hi'].to_numpy(), alpha=.16, color='#4C78A8', label='95% Wilson interval')
    ax.axhline(train_full[TARGET].mean(), color='#E45756', linestyle='--', label='Overall fraud rate')
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
    ax.set(title=f'Fraud rate by {feature}', xlabel=tick_label, ylabel='Fraction fraudulent')
    ax.legend(fontsize=9)
fig.suptitle('Temporal risk patterns: full training data', y=1.02)
finish_plot(fig, '08_time_patterns')

### 3.8 Two-variable nonlinear interactions

A heatmap shows fraud prevalence across **pairs** of quantile bins. Interaction means the relationship between one predictor and fraud may depend on another predictor. To avoid overreading small cells, the first heatmap in each row shows fraud rates and the second shows cell counts. This remains a *marginal grouped description*, not a formal interaction test.

In [ ]:
pairs = [('merchant_risk_score', 'velocity_score'), ('amount_usd', 'distance_from_home_km')]
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
interaction_tables = []
for row, (feature_a, feature_b) in enumerate(pairs):
    pair = train_full[[feature_a, feature_b, TARGET]].dropna().copy()
    for col in [feature_a, feature_b]:
        pair[col+'_bin'] = pd.qcut(pair[col], q=5, labels=False, duplicates='drop')+1
    a, b = feature_a+'_bin', feature_b+'_bin'
    grouped = pair.groupby([a,b], observed=True)[TARGET].agg(frauds='sum', n='size').reset_index()
    grouped['fraud_rate'] = grouped['frauds']/grouped['n']
    grouped['pair'] = f'{feature_a} × {feature_b}'; interaction_tables.append(grouped)
    rate_matrix = grouped.pivot(index=a,columns=b,values='fraud_rate')
    count_matrix = grouped.pivot(index=a,columns=b,values='n')
    sns.heatmap(rate_matrix*100, annot=True, fmt='.2f', cmap='rocket_r', linewidths=.6,
                cbar_kws={'label':'Fraud rate (%)'}, ax=axes[row,0])
    sns.heatmap(count_matrix, annot=True, fmt=',.0f', cmap='Blues', linewidths=.6,
                cbar_kws={'label':'Transactions per bin pair'}, ax=axes[row,1])
    axes[row,0].set(title=f'Fraud rate (%) — {feature_a} vs {feature_b}',
                    xlabel=f'{feature_b}: quintile (1=low, 5=high)', ylabel=f'{feature_a}: quintile')
    axes[row,1].set(title='Observation support for corresponding cells',
                    xlabel=f'{feature_b}: quintile (1=low, 5=high)', ylabel=f'{feature_a}: quintile')
fig.suptitle('Pairwise nonlinear diagnostics on complete labeled training set', y=1.005, fontsize=15)
finish_plot(fig, '09_bivariate_interaction_heatmaps')
pd.concat(interaction_tables, ignore_index=True).to_csv(OUTPUT_DIR / 'interaction_bin_rates.csv', index=False)

### 3.9 Potential outliers — robust univariate fences

For each numerical feature with nonzero interquartile range, flag observations outside \([Q_1-3\operatorname{IQR},\,Q_3+3\operatorname{IQR}]\). The 3×IQR rule is deliberately conservative and finds **extreme numerical values**, not fraud. Discrete features with zero IQR are skipped rather than labeling every nonzero value an outlier. We never delete flagged rows automatically.

In [ ]:
tail_features = [c for c in ['amount_usd', 'hours_since_last_txn', 'distance_from_home_km',
                   'account_balance_usd', 'velocity_score', 'merchant_risk_score'] if c in continuous_cols]
outlier_rows = []
for feature in tail_features:
    s = train_full[feature]
    q1, q3 = s.quantile([.25, .75]); iqr = q3-q1
    if iqr <= 0: continue
    lo, hi = q1 - 3*iqr, q3 + 3*iqr
    flagged = (s < lo) | (s > hi)
    outlier_rows.append({'feature': feature, 'lower_fence':lo, 'upper_fence':hi,
                         'extreme_count': int(flagged.sum()), 'extreme_pct':float(flagged.mean()),
                         'extreme_fraud_rate':float(train_full.loc[flagged, TARGET].mean()) if flagged.any() else np.nan})
outlier_stats = pd.DataFrame(outlier_rows).sort_values('extreme_pct', ascending=False)
display(outlier_stats.round(4))
outlier_stats.to_csv(OUTPUT_DIR / 'univariate_outlier_summary.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=outlier_stats, y='feature', x='extreme_pct', color='#E45756', ax=axes[0])
axes[0].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
axes[0].set(title='Share beyond conservative 3×IQR fences', xlabel='Fraction of training rows', ylabel='Numerical feature')
# Plotlog transform for visualization only: actual values are NOT transformed for this outlier test.
long_tails = eda_sample[tail_features].melt(var_name='feature', value_name='value')
sns.boxplot(data=long_tails, y='feature', x='value', color='#4C78A8', showfliers=True,
            flierprops={'marker':'.','markersize':1.5,'alpha':.3}, ax=axes[1])
axes[1].set_xscale('symlog', linthresh=1)
axes[1].set(title='Numerical distributions and tails (EDA sample)', xlabel='Feature value (symmetric log display)', ylabel='Numerical feature')
fig.suptitle('Tail diagnostics; extreme values are not automatically errors or fraud', y=1.03)
finish_plot(fig, '10_univariate_outliers')

### 3.10 Multivariate anomaly detection (unsupervised Isolation Forest)

**Outlier:** an observation unlike most other observations in the modeled feature space. **Fraud:** a supervised target label. These concepts can overlap but are not equivalent.

An `IsolationForest` partitions feature space randomly; points separated in fewer splits tend to have higher anomaly scores. This analysis fits on a **fixed-seed random subset**, **without the target or identifier**. It uses one-hot categorical features and log1p-transformed nonnegative numerical values. Its `contamination=0.02` is an arbitrary *demonstration threshold*, not an estimated fraud prevalence. It is an EDA diagnostic, not a fraud classifier and not part of the CV scoring pipeline. We plot class-conditional anomaly-score distributions and observed fraud rates by anomaly-score decile.

In [ ]:
anomaly_view = train_full.sample(n=min(ANOMALY_SAMPLE_SIZE, len(train_full)), random_state=RANDOM_STATE).copy()
anomaly_matrix = anomaly_view.drop(columns=[ID, TARGET]).copy()
for feature in continuous_cols:
    if anomaly_matrix[feature].min() >= 0:
        anomaly_matrix[feature] = np.log1p(anomaly_matrix[feature].astype(float))
anomaly_matrix = pd.get_dummies(anomaly_matrix, columns=categorical_cols, dtype=np.int8)
anomaly_matrix = anomaly_matrix.fillna(anomaly_matrix.median(numeric_only=True))
iso = IsolationForest(n_estimators=120, contamination=.02, max_samples=256,
                      n_jobs=N_JOBS, random_state=RANDOM_STATE)
iso.fit(anomaly_matrix)   # Unsupervised: labels are not part of training inputs.
anomaly_view['anomaly_score'] = -iso.decision_function(anomaly_matrix)
anomaly_view['anomaly_flag'] = anomaly_view['anomaly_score'] > 0
anomaly_view['transaction_class'] = anomaly_view[TARGET].map({0:'Legitimate',1:'Fraud'})
flagged_summary = anomaly_view.groupby('anomaly_flag')[TARGET].agg(['sum','size','mean'])
print('Anomaly-flag association (descriptive; scored on same sample used to fit detector):')
display(flagged_summary)
anomaly_view['anomaly_decile'] = pd.qcut(anomaly_view['anomaly_score'], q=10, labels=False, duplicates='drop')+1
by_decile = anomaly_view.groupby('anomaly_decile')[TARGET].agg(['sum','size']).reset_index()
by_decile['rate'] = by_decile['sum']/by_decile['size']
by_decile['lo'], by_decile['hi'] = wilson_bounds(by_decile['sum'], by_decile['size'])
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=anomaly_view, x='anomaly_score', hue='transaction_class',
             stat='density', common_norm=False, element='step', fill=False,
             bins=45, palette={'Legitimate':'#4C78A8','Fraud':'#E45756'}, ax=axes[0])
axes[0].axvline(0, color='black', ls='--', label='2% detector threshold')
axes[0].set(title='Anomaly-score distributions by target class', xlabel='Higher = more anomalous (Isolation Forest)', ylabel='Density within each class')
axes[0].legend(handles=[plt.Line2D([0],[0],color='#4C78A8',label='Legitimate'),
                        plt.Line2D([0],[0],color='#E45756',label='Fraud'),
                        plt.Line2D([0],[0],color='black',ls='--',label='Anomaly threshold')], fontsize=9)
sns.barplot(data=by_decile, x='anomaly_decile', y='rate', color='#4C78A8', ax=axes[1])
for i, row in by_decile.iterrows(): axes[1].plot([i,i],[row.lo,row.hi], color='#253858', lw=1.8)
axes[1].axhline(train_full[TARGET].mean(), color='#E45756', ls='--', label='Overall fraud rate')
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
axes[1].set(title='Fraud rate by anomaly-score decile', xlabel='Anomaly decile (10 = most anomalous)', ylabel='Observed fraud rate')
axes[1].legend()
fig.suptitle('Unsupervised anomaly diagnostic on random 30k-row subset', y=1.03)
finish_plot(fig, '11_isolation_forest_anomaly_diagnostics')
display(anomaly_view.nlargest(8, 'anomaly_score')[[ID, TARGET, 'anomaly_score',
          'amount_usd', 'velocity_score', 'merchant_risk_score', 'distance_from_home_km']])
by_decile.to_csv(OUTPUT_DIR / 'anomaly_decile_rates.csv', index=False)

### 3.11 Which single predictors discriminate fraud?

A **univariate ROC AUC** of one feature alone measures how well it ranks fraud versus legitimate transactions, without adjusting for other features. We report `max(AUC,1−AUC)` solely as *direction-agnostic screening strength*. It does not measure causal importance, multivariate added value, or statistical significance. To keep this exploratory plot responsive, it uses the fixed-seed `eda_sample` rather than all 699,960 records. A single-feature AUC can be below 0.5 if its direction is reversed.

In [ ]:
single_auc_rows = []
for feature in continuous_cols + boolean_cols:
    values = pd.to_numeric(eda_sample[feature], errors='coerce')
    mask = values.notna() & eda_sample[TARGET].notna()
    if values.loc[mask].nunique() < 2 or eda_sample.loc[mask, TARGET].nunique() < 2:
        continue
    auc_value = roc_auc_score(eda_sample.loc[mask, TARGET].astype(int), values.loc[mask])
    single_auc_rows.append({'feature':feature, 'univariate_auc':auc_value,
                            'direction_agnostic_auc':max(auc_value, 1-auc_value),
                            'direction':'Higher = more fraud' if auc_value>=.5 else 'Lower = more fraud',
                            'n':int(mask.sum())})
single_feature_auc = pd.DataFrame(single_auc_rows).sort_values('direction_agnostic_auc',ascending=False)
single_feature_auc.to_csv(OUTPUT_DIR/'eda_univariate_auc.csv', index=False)
fig,ax=plt.subplots(figsize=(12,max(6,.34*len(single_feature_auc))))
sns.barplot(data=single_feature_auc,x='direction_agnostic_auc',y='feature',hue='direction',dodge=False,ax=ax)
ax.axvline(.5,color='gray',ls='--',label='Random direction (0.5)')
ax.set(title=f'Single-feature fraud ranking | random sample n={len(eda_sample):,}',
       xlabel='Direction-agnostic univariate ROC AUC (higher = stronger ranking)',ylabel='Predictor')
ax.set_xlim(.48,1.01);ax.legend(title='Observed marginal direction',loc='lower right',fontsize=9)
finish_plot(fig,'20_univariate_feature_discrimination')

### 3.12 Class-conditional empirical cumulative distributions

The **empirical CDF**, $F(t)=n^{-1}\sum_i\mathbf{1}(X_i\le t)$, helps reveal shifts and tail differences without choosing histogram bins. Sampling equal numbers per class helps display the rare fraud class, so these curves are **within-class distributions**, not population fraud rates.

In [ ]:
ecdf_features=[c for c in ['amount_usd','hours_since_last_txn','velocity_score','merchant_risk_score'] if c in continuous_cols]
balanced_ecdf=pd.concat([train_full[train_full[TARGET]==v].sample(n=min(2500,(train_full[TARGET]==v).sum()),random_state=RANDOM_STATE) for v in [0,1]],ignore_index=True)
fig,axs=plt.subplots(2,2,figsize=(16,10))
for ax,col in zip(axs.flat,ecdf_features):
    sns.ecdfplot(data=balanced_ecdf,x=col,hue=TARGET,hue_order=[0,1],palette=['#4C78A8','#E45756'],ax=ax)
    ax.set(title=f'Empirical CDF by observed class: {col}',xlabel=col,ylabel='Within-class cumulative share')
    if col in ('amount_usd','hours_since_last_txn') and balanced_ecdf[col].min()>=0:
        ax.set_xscale('symlog',linthresh=1)
    leg=ax.get_legend()
    if leg: leg.set_title('Observed class (0 = legitimate, 1 = fraud)')
for ax in axs.flat[len(ecdf_features):]:ax.set_visible(False)
fig.suptitle('Distribution shifts between fraud and legitimate transactions | equal-class sample',y=1.01)
finish_plot(fig,'21_class_conditional_ecdfs')

### 3.13 Conditional fraud rates across categorical × Boolean groups

A categorical risk association can change when a binary flag is present. The heatmap shows **observed fraud rate**, not causal effect. We annotate support; tiny cells should not drive conclusions. Rates are based on the complete labeled training dataset.

In [ ]:
group_cats=[c for c in ['channel','auth_method'] if c in categorical_cols]
group_flags=[c for c in ['is_foreign_transaction','used_vpn'] if c in boolean_cols]
fig,axs=plt.subplots(len(group_cats),len(group_flags),figsize=(15,9),squeeze=False)
conditional_group_rows=[]
for i,cat in enumerate(group_cats):
    for j,flag in enumerate(group_flags):
        grouped=train_full.groupby([cat,flag],observed=True)[TARGET].agg(n='size',frauds='sum').reset_index()
        grouped['rate']=100*grouped['frauds']/grouped['n']
        grouped['cat_feature']=cat;grouped['flag_feature']=flag
        conditional_group_rows.append(grouped.rename(columns={cat:'category_value',flag:'flag_value'}))
        rates=grouped.pivot(index=cat,columns=flag,values='rate')
        supports=grouped.pivot(index=cat,columns=flag,values='n')
        sns.heatmap(rates,ax=axs[i,j],annot=True,fmt='.2f',cmap='mako',linewidths=.5,
                    cbar_kws={'label':'Observed fraud rate (%)'})
        axs[i,j].set(title=f'{cat} × {flag}',xlabel=f'{flag} (Boolean)',ylabel=cat)
        for t in axs[i,j].texts:
            xx,yy=int(t.get_position()[0]-.5),int(t.get_position()[1]-.5)
            n=supports.iloc[yy,xx]
            t.set_text(f'{t.get_text()}%\nn={n:,.0f}' if pd.notna(n) else 'n/a')
conditional_groups=pd.concat(conditional_group_rows,ignore_index=True)
conditional_groups.to_csv(OUTPUT_DIR/'eda_category_flag_rates.csv',index=False)
fig.suptitle('Observed fraud prevalence and support across feature combinations | all training rows',y=1.015)
finish_plot(fig,'22_categorical_flag_interactions')

### 3.14 Weekly × hourly risk surface

Hour-of-day and day-of-week are cyclical. The surface visualizes observed prevalence, while the separate support matrix makes sparse cells visible. These are temporal **descriptive** patterns, not evidence of temporal independence or a validation of sequential random folds.

In [ ]:
temporal_cols=['day_of_week','time_of_day_hour']
if all(c in train_full.columns for c in temporal_cols):
    surface=train_full.groupby(temporal_cols)[TARGET].agg(n='size',frauds='sum').reset_index()
    surface['pct']=surface.frauds/surface.n*100
    hours=np.arange(24);days=np.arange(7)
    pivot_rate=surface.pivot(index='day_of_week',columns='time_of_day_hour',values='pct').reindex(index=days,columns=hours)
    pivot_n=surface.pivot(index='day_of_week',columns='time_of_day_hour',values='n').reindex(index=days,columns=hours)
    fig,axs=plt.subplots(2,1,figsize=(17,9))
    sns.heatmap(pivot_rate,cmap='rocket_r',ax=axs[0],cbar_kws={'label':'Fraud prevalence (%)'})
    sns.heatmap(pivot_n,cmap='Blues',ax=axs[1],cbar_kws={'label':'Transaction count'})
    axs[0].set(title='Fraud frequency by weekday and hour | all training rows',xlabel='Hour of day (0–23)',ylabel='Day of week (0–6)')
    axs[1].set(title='Cell support for interpreting the prevalence heatmap',xlabel='Hour of day (0–23)',ylabel='Day of week (0–6)')
    finish_plot(fig,'23_weekly_hourly_fraud_surface')
    surface.to_csv(OUTPUT_DIR/'eda_weekly_hourly_rates.csv',index=False)

### 3.15 Multivariate geometry: pairwise distributions

This pairplot uses a **class-balanced display sample**, intentionally oversampling fraud for legibility. Therefore it cannot be used to estimate prevalence or overall correlation magnitudes. Diagonal KDE plots show within-class density; off-diagonal scatters show overlap and possible nonlinear structure.

In [ ]:
pair_features=[c for c in ['amount_usd','velocity_score','merchant_risk_score','hours_since_last_txn'] if c in continuous_cols]
pair_view=pd.concat([train_full[train_full[TARGET]==label].sample(n=min(550,(train_full[TARGET]==label).sum()),random_state=RANDOM_STATE+11) for label in [0,1]],ignore_index=True)
if len(pair_features)>=2:
    pp=sns.pairplot(pair_view[pair_features+[TARGET]],vars=pair_features,hue=TARGET,
                    hue_order=[0,1],palette=['#4C78A8','#E45756'],diag_kind='hist',
                    corner=True,plot_kws={'alpha':.38,'s':10,'rasterized':True},
                    diag_kws={'bins':35,'alpha':.55},height=2.4)
    pp.fig.suptitle('Pairwise structure by class | balanced display sample, not prevalence',y=1.02)
    pp.fig.tight_layout()
    pp.fig.savefig(FIGURE_DIR/'24_pairwise_class_geometry.png',dpi=160,bbox_inches='tight')
    plt.show();plt.close(pp.fig)

### 3.16 Train-versus-test feature shift

Kaggle test labels are unknown. Comparing *unlabeled* feature distributions helps detect whether the test sample looks different from training. The **Kolmogorov–Smirnov (KS) statistic** measures maximal CDF difference for one numerical feature; it does not establish equivalence, and multiple comparisons are not adjusted.

In [ ]:
shift_cols=[c for c in ['amount_usd','velocity_score','merchant_risk_score','hours_since_last_txn','distance_from_home_km','account_balance_usd'] if c in continuous_cols and c in test.columns]
tr_s=train_full.sample(n=min(12000,len(train_full)),random_state=RANDOM_STATE)
te_s=test.sample(n=min(12000,len(test)),random_state=RANDOM_STATE)
shift_rows=[]
fig,axs=plt.subplots(2,3,figsize=(17,9))
for ax,feature in zip(axs.flat,shift_cols):
    a=tr_s[feature].dropna();b=te_s[feature].dropna()
    ks=float(ks_2samp(a,b).statistic)
    shift_rows.append({'feature':feature,'sample_train':len(a),'sample_test':len(b),'ks_statistic':ks})
    sns.ecdfplot(x=a,ax=ax,label='Train',color='#4C78A8')
    sns.ecdfplot(x=b,ax=ax,label='Test (unlabeled)',color='#E45756')
    ax.set(title=f'{feature} | KS={ks:.3f}',xlabel=feature,ylabel='Cumulative share')
    ax.legend(title='Dataset')
for ax in axs.flat[len(shift_cols):]:ax.set_visible(False)
fig.suptitle('Unlabeled feature shift: train vs Kaggle test | equal-size random samples',y=1.01)
finish_plot(fig,'25_train_test_ecdf_shift')
pd.DataFrame(shift_rows).to_csv(OUTPUT_DIR/'eda_feature_shift_ecdf.csv',index=False)

### 3.17 Interpretation and hypothesis generation

These graphics are **screening devices**. For the report, state which binned profiles look nonlinear, which interaction cells have adequate sample support, and whether unusually large transactions or multivariate anomalies exhibit elevated fraud rates. A heatmap alone is not a significance test. Do not remove outliers just because their values are extreme; excluding genuine fraud patterns may damage detection performance. A subsequent experiment can compare baseline logistic regression with spline logistic regression and nonlinear tree models under identical stratified CV.

**Methodological caution:** Inspecting the full training set to choose spline knots or features is useful for discovery, but can create selection bias. The OOF results are not a fully unbiased estimate when modeling choices are informed by all labels. Prefer a completely untouched holdout for the final performance claim. The fixed spline feature set is declared before model CV, and all spline coefficients/knots are fit within folds.


## 4. SVD/PCA: how many dimensions explain 90% of the feature variance?

For a centered, preprocessed matrix $Z$, let $Z=U\Sigma V^\top$. The variance share explained by the first $k$ PCs is

$$r(k)=\frac{\sum_{j=1}^{k}\sigma_j^2}{\sum_{j=1}^{p}\sigma_j^2}.$$

This is a **diagnostic sample** only. The predictive PCA pipeline fitted below repeats preprocessing and PCA *within each training fold*; this prevents validation leakage. We do not apply PCA automatically to tree-based models.

## 5. Why logistic regression? Construct model pipelines

**Logistic regression** is the baseline because (i) it is fast on large tabular data, (ii) its L2 regularization helps control variance, (iii) its coefficients describe *additive conditional log-odds* after preprocessing, and (iv) it yields a ranking score suitable for ROC AUC. For \(x\in\mathbb R^p\), \(\Pr(Y=1\mid x)=\sigma(\beta_0+\beta^\top x)\), where \(\sigma(t)=(1+e^{-t})^{-1}\). **The probability is sigmoidal, but the log-odds are linear in the features.** Interpret standardized numerical coefficients as changes in log-odds per standard deviation, while categorical coefficients are relative to their encoded reference level. Correlation among predictors, regularization and class weights still complicate coefficient interpretation; association is not causation.

**Rival models:** PCA90 + logistic tests unsupervised compression; **spline logistic** applies B-spline basis expansion to selected numeric features so \(\beta^\top\phi(x)\) may be curved (but still additive unless interaction features are explicitly included); random forest and XGBoost learn nonlinear thresholds and interactions. The spline model is an experiment motivated by the nonlinear-rate profiles—not a presumption of improved AUC.

All learned feature transforms are fit **inside each CV training fold**. Standard logistic and PCA logistic use `StandardScaler` for numerical columns. Spline logistic uses `SplineTransformer` for three specified numerical columns; tree models retain the original numeric and one-hot columns. No outlier flags or exclusions are applied to the training data.


In [ ]:
# Use the entire train set by default; a stratified development sample is only for smoke testing.
if FAST_DEV_RUN:
    if DEV_SAMPLE_SIZE >= len(train_full):
        train = train_full.copy()
    else:
        train, _ = train_test_split(train_full, train_size=DEV_SAMPLE_SIZE,
                                    stratify=train_full[TARGET], random_state=RANDOM_STATE)
    train = train.reset_index(drop=True)
else:
    train = train_full

features = [c for c in train.columns if c not in (ID, TARGET)]
categorical_cols = train[features].select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = [c for c in features if c not in categorical_cols]
X = train[features]
y = train[TARGET].astype(int)
X_test = test[features]
assert len(X) == len(y) and y.nunique() == 2

# OneHotEncoder(sparse_output=False) makes dense output: avoid sparse-0-as-missing
# discrepancies in XGBoost, at the expense of additional RAM.
SPLINE_FEATURES = [c for c in ('amount_usd', 'velocity_score', 'merchant_risk_score') if c in numeric_cols]

def make_preprocessor(scale_numeric: bool, spline_numeric: bool = False,
                      drop_reference: bool = False) -> ColumnTransformer:
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    transformers = []
    if spline_numeric:
        # B-splines allow additive nonlinear log-odds, not arbitrary interactions.
        transformers.append(('spline', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('spline', SplineTransformer(n_knots=5, degree=3, include_bias=False,
                                         extrapolation='constant')),
            ('scaler', StandardScaler()),
        ]), SPLINE_FEATURES))
        numeric_remainder = [col for col in numeric_cols if col not in SPLINE_FEATURES]
    else:
        numeric_remainder = numeric_cols
    transformers.extend([
            ('numeric', Pipeline(numeric_steps), numeric_remainder),
            ('categorical', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore',
                                         drop='first' if drop_reference else None,
                                         sparse_output=False,
                                         dtype=np.float32))
            ]), categorical_cols)
        ])
    return ColumnTransformer(
        transformers=transformers,
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )

class LogisticOffsetXGBClassifier(ClassifierMixin, BaseEstimator):
    """Frozen logistic-logit offset + boosted-tree correction.

    Input X is ALREADY preprocessed by a training-fold Pipeline. Important:
    both fit and predict_proba must pass corresponding raw LR logits to
    XGBClassifier as base_margin; otherwise the model is mis-specified.
    """
    def __init__(self, C=1.0, n_estimators=160, max_depth=2,
                 learning_rate=0.05, min_child_weight=5,
                 n_jobs=-1, random_state=42):
        self.C = C
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.min_child_weight = min_child_weight
        self.n_jobs = n_jobs
        self.random_state = random_state

    def fit(self, X, y):
        self.logistic_ = LogisticRegression(
            C=self.C, solver='lbfgs', max_iter=600,
            random_state=self.random_state,
        ).fit(X, y)
        y = np.asarray(y, dtype=np.int32)
        self.classes_ = self.logistic_.classes_
        if not np.array_equal(self.classes_, [0, 1]):
            raise ValueError('This estimator requires both binary labels 0 and 1.')
        train_logits = np.asarray(self.logistic_.decision_function(X), dtype=np.float32)
        self.booster_ = XGBClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            min_child_weight=self.min_child_weight,
            subsample=0.85, colsample_bytree=0.85,
            objective='binary:logistic', eval_metric='logloss',
            tree_method='hist', n_jobs=self.n_jobs,
            random_state=self.random_state,
        )
        # The XGBoost objective learns a correction starting at LR's *logits*.
        self.booster_.fit(X, y, base_margin=train_logits, verbose=False)
        return self

    def predict_proba(self, X):
        check_is_fitted(self, ['logistic_', 'booster_'])
        logits = np.asarray(self.logistic_.decision_function(X), dtype=np.float32)
        return self.booster_.predict_proba(X, base_margin=logits)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    def predict_components(self, X):
        """Return baseline probabilities, corrected probabilities and logit delta."""
        check_is_fitted(self, ['logistic_', 'booster_'])
        baseline_logits = np.asarray(self.logistic_.decision_function(X), dtype=np.float64)
        baseline_prob = self.logistic_.predict_proba(X)[:, 1]
        corrected_prob = self.predict_proba(X)[:, 1].astype(np.float64)
        corrected_clipped = np.clip(corrected_prob, 1e-7, 1 - 1e-7)
        corrected_logits = np.log(corrected_clipped / (1 - corrected_clipped))
        return baseline_prob, corrected_prob, corrected_logits - baseline_logits


def make_pipeline(spec: dict) -> Pipeline:
    family = spec['family']
    params = spec['params']
    if family in ('logistic', 'logistic_pca', 'logistic_spline'):
        model = LogisticRegression(
            C=params['C'], class_weight=params.get('class_weight'),
            max_iter=600, solver='lbfgs', random_state=RANDOM_STATE
        )
        steps = [('preprocess', make_preprocessor(scale_numeric=True, spline_numeric=(family == 'logistic_spline'),
                                                        drop_reference=True))]
        if family == 'logistic_pca':
            steps.append(('pca', PCA(n_components=0.90, svd_solver='full')))
    elif family == 'logistic_offset_xgb':
        # One fold-safe preprocessing transform feeds both stages.
        model = LogisticOffsetXGBClassifier(
            C=params['C'], n_estimators=params['n_estimators'],
            max_depth=params['max_depth'], learning_rate=params['learning_rate'],
            min_child_weight=params['min_child_weight'],
            n_jobs=N_JOBS, random_state=RANDOM_STATE,
        )
        steps = [('preprocess', make_preprocessor(scale_numeric=True,
                    drop_reference=True))]
    elif family == 'random_forest':
        model = RandomForestClassifier(
            n_estimators=params['n_estimators'], max_depth=params['max_depth'],
            min_samples_leaf=params['min_samples_leaf'],
            max_samples=0.75, class_weight='balanced_subsample',
            max_features='sqrt', n_jobs=N_JOBS, random_state=RANDOM_STATE,
        )
        steps = [('preprocess', make_preprocessor(scale_numeric=False))]
    elif family == 'xgboost':
        model = XGBClassifier(
            n_estimators=params['n_estimators'], max_depth=params['max_depth'],
            learning_rate=params['learning_rate'], min_child_weight=params['min_child_weight'],
            subsample=0.85, colsample_bytree=0.85,
            objective='binary:logistic', eval_metric='auc', tree_method='hist',
            n_jobs=N_JOBS, random_state=RANDOM_STATE,
        )
        steps = [('preprocess', make_preprocessor(scale_numeric=False))]
    else:
        raise ValueError(f'Unknown model family: {family}')
    return Pipeline(steps + [('model', model)])

# Small, interpretable hyperparameter grids: every combination is retained and
# receives five independent training-fold fits. Expand only if time permits.
specs = [
    {'name':'logit_C0p1', 'family':'logistic', 'params':{'C':0.1}},
    {'name':'logit_C1', 'family':'logistic', 'params':{'C':1.0}},
    {'name':'logit_balanced_C1', 'family':'logistic', 'params':{'C':1.0, 'class_weight':'balanced'}},
    {'name':'logit_spline_C0p1', 'family':'logistic_spline', 'params':{'C':0.1}},
    {'name':'logit_spline_C1', 'family':'logistic_spline', 'params':{'C':1.0}},
    {'name':'pca90_logit_C0p1', 'family':'logistic_pca', 'params':{'C':0.1}},
    {'name':'pca90_logit_C1', 'family':'logistic_pca', 'params':{'C':1.0}},
    {'name':'offset_xgb_depth2_lr05', 'family':'logistic_offset_xgb',
     'params':{'C':1.0, 'n_estimators':160, 'max_depth':2,
               'learning_rate':0.05, 'min_child_weight':5}},
    {'name':'offset_xgb_depth3_lr05', 'family':'logistic_offset_xgb',
     'params':{'C':1.0, 'n_estimators':240, 'max_depth':3,
               'learning_rate':0.05, 'min_child_weight':5}},
    {'name':'rf_depth12_leaf10', 'family':'random_forest', 'params':{'n_estimators':120, 'max_depth':12, 'min_samples_leaf':10}},
    {'name':'rf_depth18_leaf5', 'family':'random_forest', 'params':{'n_estimators':180, 'max_depth':18, 'min_samples_leaf':5}},
    {'name':'rf_depthNone_leaf15', 'family':'random_forest', 'params':{'n_estimators':160, 'max_depth':None, 'min_samples_leaf':15}},
    {'name':'xgb_depth3_lr08', 'family':'xgboost', 'params':{'n_estimators':250, 'max_depth':3, 'learning_rate':0.08, 'min_child_weight':5}},
    {'name':'xgb_depth5_lr05', 'family':'xgboost', 'params':{'n_estimators':350, 'max_depth':5, 'learning_rate':0.05, 'min_child_weight':5}},
    {'name':'xgb_depth6_lr04', 'family':'xgboost', 'params':{'n_estimators':400, 'max_depth':6, 'learning_rate':0.04, 'min_child_weight':10}},
]

if MAX_CANDIDATES_PER_FAMILY is not None:
    seen, trimmed = {}, []
    for spec in specs:
        family = spec['family']
        if seen.get(family, 0) < MAX_CANDIDATES_PER_FAMILY:
            trimmed.append(spec)
            seen[family] = seen.get(family, 0) + 1
    specs = trimmed

# Keep FAST_DEV_RUN inexpensive without changing full-run hyperparameters.
if FAST_DEV_RUN:
    for spec in specs:
        if spec['family'] == 'random_forest':
            spec['params']['n_estimators'] = min(45, spec['params']['n_estimators'])
        if spec['family'] == 'logistic_offset_xgb':
            spec['params']['n_estimators'] = min(45, spec['params']['n_estimators'])
        if spec['family'] == 'xgboost':
            spec['params']['n_estimators'] = min(70, spec['params']['n_estimators'])
print('CV candidates:', len(specs), 'training rows:', len(X))
display(pd.DataFrame(specs))

### 5.1 Logistic regression assumptions: what is required, what is checked, and what is unknown

**Purpose.** Logistic regression is a *discriminative* model of a binary conditional response, not an ordinary least-squares regression. Its conditional likelihood assumes

$$Y_i\mid X_i=x_i\sim\operatorname{Bernoulli}(p_i),\qquad
\log\frac{p_i}{1-p_i}=\beta_0+x_i^\top\beta.$$

**Importantly, neither normally distributed features nor homoscedastic residuals are required.** The Bernoulli conditional variance is $\operatorname{Var}(Y_i\mid X_i)=p_i(1-p_i)$.

| Condition | Why it matters | What we can conclude from this dataset |
|:--|:--|:--|
| **1. Binary target** | A binary Bernoulli likelihood is appropriate only for a two-class outcome. | **Verified**: `is_fraud` contains 0 and 1. |
| **2. Independent sampling / dependence modeled correctly** | Standard independent-row likelihood and ordinary CV uncertainty may be invalid for repeated customers, merchants, or clustered/time-dependent transactions. | **Unverifiable**: no customer/group identifier or reliable temporal ordering. Unique transaction `id` is not proof of independence. |
| **3. Appropriate functional form** | *Standard* logistic regression assumes the **conditional log-odds** are additive and linear in the encoded covariates, not that raw fraud probability is linear. | **Questionable**: marginal nonlinear curves motivate splines; paired OOF model comparisons below check whether nonlinear structure improves prediction, not a definitive specification test. |
| **4. No severe multicollinearity / exact redundancy** | Exact collinearity obstructs unique unpenalized coefficient estimates; high collinearity destabilizes interpretation. | We use **`drop='first'`** for categorical variables in logistic pipelines (including PCA/splines) to avoid the dummy-variable trap; tree pipelines may retain all indicators. Numerical VIFs below are a *screening diagnostic*, not a guarantee that the full encoded design is well-conditioned. L2 penalties stabilize predictions but do not eliminate interpretability issues. |
| **5. No complete or quasi-complete separation** | If classes separate perfectly, unpenalized maximum-likelihood coefficients can diverge. | Check one-variable category/flag levels below; **no marginal separation does not rule out multivariate separation**. L2-penalized logistic regression remains finite. |
| **6. Sufficient positive events** | Extremely rare events, very sparse categories, or a large number of fitted parameters can create unstable estimates. | **11,689 frauds in 699,960 training cases**; evaluate **smallest** level counts and number of coefficients rather than applying an arbitrary universal 10-events-per-variable cutoff. |
| **7. No dominating/influential cases** | Extreme *feature combinations* may strongly influence fitted log-odds. | The 3×IQR and Isolation Forest sections flag unusual transactions. This is **not** a formal influence or leverage test; unusual observations are not automatically errors and are retained. |
| **8. Representative labeled training data / comparable test distribution** | A model fitted on one population need not generalize to another. | Screen **feature distribution shift** below; unlabeled Kaggle test data cannot establish whether $P(Y\mid X)$ is stable. |
| **9. Convergence of numerical optimization** | Without convergence, the estimated coefficients may not be reliable. | Each fitted logistic CV fold records iteration count and a `ConvergenceWarning` flag in `fold_metrics.csv`. |
| **10. Well-calibrated probabilities** | This matters for decisions and interpretation of probabilities, but **calibration is not a prerequisite for ROC AUC ranking**. | Assess OOF reliability curves, Brier score, and log loss after CV. Class-weighted logistic regression especially needs separate calibration scrutiny. |

**Scope:** These checks apply mainly to **logistic regression**. Random forests and XGBoost do not require linear log-odds or absence of collinearity for fitting, but still need representative evaluation, trustworthy labels, leakage control, and sufficient generalization. The `id` column is deliberately excluded.

**Assumption status is not a single pass/fail test.** A model can be useful for risk ranking even when its additive log-odds specification is imperfect, but its coefficients and probability estimates then deserve more caution. Descriptive EDA or large-sample significance alone cannot certify the assumptions.

In [ ]:
# Reproducible *screening* diagnostics on observed data (no model fitting on Kaggle test labels).
assert y.isin([0, 1]).all()
feature_duplicates = int(train_full.drop(columns=[TARGET, ID]).duplicated().sum())
model_diagnostic_sample = X.sample(n=min(ASSUMPTION_SAMPLE_SIZE, len(X)),
                                   random_state=RANDOM_STATE)
num_audit_cols = [c for c in numeric_cols if pd.api.types.is_numeric_dtype(X[c])
                  or pd.api.types.is_bool_dtype(X[c])]
num_audit_matrix = model_diagnostic_sample[num_audit_cols].astype(float)
stds = num_audit_matrix.std(ddof=0)
nonconstant = stds[stds > 1e-10].index.tolist()
Z_audit = (num_audit_matrix[nonconstant] - num_audit_matrix[nonconstant].mean()) / stds[nonconstant]
R_audit = np.corrcoef(Z_audit.to_numpy(), rowvar=False)
condition_numeric = float(np.linalg.cond(R_audit))
if condition_numeric < 1e10:
    vif_values = np.diag(np.linalg.inv(R_audit))
else:
    vif_values = [float('inf')] * len(nonconstant)
vif_table = pd.DataFrame({'feature': nonconstant, 'vif': vif_values})\
    .sort_values('vif', ascending=False).reset_index(drop=True)

# Marginal separation and sparse levels: every observed level should contain both classes.
separation_rows = []
for col in categorical_cols + boolean_cols:
    rates = train_full.groupby(col, dropna=False, observed=True)[TARGET].agg(['size', 'sum'])
    for level, info in rates.iterrows():
        separation_rows.append({'variable':col, 'level':str(level),
                                'count':int(info['size']), 'fraud_count':int(info['sum']),
                                'all_one_class':bool(info['sum'] == 0 or info['sum'] == info['size'])})
separation_table = pd.DataFrame(separation_rows)

# Population overlap screening: KS D (numerical; max CDF distance) and categorical
# total variation (half the L1 distance of category probabilities). These are
# effect sizes, not proofs that future P(Y|X) is stable.
train_shift = train_full.sample(n=min(SHIFT_SAMPLE_SIZE, len(train_full)), random_state=RANDOM_STATE)
test_shift = test.sample(n=min(SHIFT_SAMPLE_SIZE, len(test)), random_state=RANDOM_STATE)
shift_rows = []
for col in continuous_cols + boolean_cols:
    a = train_shift[col].astype(float).dropna().to_numpy()
    b = test_shift[col].astype(float).dropna().to_numpy()
    shift_rows.append({'feature':col, 'type':'numeric/flag',
                       'distribution_distance':float(ks_2samp(a, b).statistic)})
for col in categorical_cols:
    a = train_shift[col].value_counts(normalize=True)
    b = test_shift[col].value_counts(normalize=True)
    levels = a.index.union(b.index)
    tv = float(0.5*np.abs(a.reindex(levels, fill_value=0).to_numpy() -
                          b.reindex(levels, fill_value=0).to_numpy()).sum())
    shift_rows.append({'feature':col, 'type':'categorical',
                       'distribution_distance':tv})
shift_table = pd.DataFrame(shift_rows).sort_values('distribution_distance', ascending=False)

# Store only aggregate summaries (not individual competition records) outside ignored outputs.
vif_table.to_csv(OUTPUT_DIR/'assumptions_numeric_vif.csv', index=False)
separation_table.to_csv(OUTPUT_DIR/'assumptions_marginal_levels.csv', index=False)
shift_table.to_csv(OUTPUT_DIR/'assumptions_train_test_shift.csv', index=False)

print('Unique feature rows:', len(train_full)-feature_duplicates, '/', len(train_full))
print('Missing predictor values:', int(train_full[features].isna().sum().sum()))
print('All-one-class marginal categorical/flag levels:', int(separation_table.all_one_class.sum()),
      '/', len(separation_table))
print('Numerical correlation matrix condition number:', round(condition_numeric, 2))
print('Numerical-only VIF > 5:', int((vif_table.vif > 5).sum()))
display(vif_table.head(12))
display(shift_table.head(12))

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
selection = vif_table.head(12).sort_values('vif')
sns.barplot(data=selection, x='vif', y='feature', color='#4C78A8', ax=axes[0])
axes[0].axvline(5, color='#E45756', linestyle='--', label='Screening guide: VIF = 5')
axes[0].set(title='Collinearity screening: numerical/Boolean predictors',
            xlabel='Variance inflation factor (VIF)', ylabel='Predictor')
axes[0].legend(loc='best')
for kind, color in [('numeric/flag','#4C78A8'),('categorical','#F58518')]:
    subset = shift_table.head(12)
    subset = subset[subset.type == kind]
    axes[1].barh(subset.feature, subset.distribution_distance,
                 label='KS D (numerical/flag)' if kind=='numeric/flag' else 'Total variation (categorical)', color=color)
axes[1].invert_yaxis()
axes[1].set(title='Train–test covariate-shift screening (sampled)',
            xlabel='Distribution distance (0 = identical empirical distribution)', ylabel='Feature')
axes[1].legend(loc='best')
finish_plot(fig, '17_assumptions_vif_and_feature_shift')

# A concise, reproducible evidence summary; no unsupported global "all passed" statement.
assumption_evidence = {
    'n_train': int(len(train_full)),
    'fraud_events': int(train_full[TARGET].sum()),
    'prevalence': float(train_full[TARGET].mean()),
    'predictor_missing_values': int(train_full[features].isna().sum().sum()),
    'duplicate_feature_rows': feature_duplicates,
    'pure_outcome_marginal_levels': int(separation_table.all_one_class.sum()),
    'minimum_marginal_level_count': int(separation_table['count'].min()),
    'max_numeric_vif': float(vif_table.vif.max()),
    'numeric_correlation_condition_number': condition_numeric,
    'max_train_test_shift_distance': float(shift_table.distribution_distance.max()),
    'independence_verifiable': False,
    'full_multivariate_separation_tested': False,
    'influential_case_formal_tested': False,
}
(OUTPUT_DIR/'assumptions_evidence.json').write_text(json.dumps(assumption_evidence,indent=2))
display(pd.DataFrame([assumption_evidence]).T.rename(columns={0:'diagnostic'}))

### 5.2 Boosting the errors of logistic regression: an offset XGBoost model

**Research question.** Can shallow gradient-boosted trees improve on a *fixed, interpretable logistic regression* by learning nonlinear corrections to its log-odds? This is **not** probability bagging, and it is not ordinary least-squares regression on the binary residuals.

Logistic regression first fits the baseline score \(f_0(x)=\beta_0+\beta^\top x\). If \(\sigma(t)=1/(1+e^{-t})\), the resulting predicted fraud probability is \(p_0(x)=\sigma(f_0(x))\). An XGBoost model then learns additive corrections \(g(x)=\sum_{m=1}^M\eta h_m(x)\), producing

$$\boxed{\hat p(x)=\sigma\!\left(f_0(x)+g(x)\right)}.$$

For one binary observation the negative log likelihood and its derivative with respect to the **logit** \(f\) are

$$\ell(y,f)=-y\log\sigma(f)-(1-y)\log(1-\sigma(f)),\quad \frac{\partial\ell}{\partial f}=\sigma(f)-y,\quad \frac{\partial^2\ell}{\partial f^2}=\sigma(f)(1-\sigma(f)).$$

Thus the negative gradient is \(y-p\). XGBoost's `binary:logistic` objective uses this gradient and Hessian when fitting the *correction trees*; we provide \(f_0(x)\) as the `base_margin`. **Never pass probabilities as `base_margin`: it expects raw logits.** Unlike fitting a residual regressor to \(y-p\), this procedure respects the Bernoulli log loss and keeps probabilities in \([0,1]\).

**Fold safety:** inside each outer CV fold, fit the preprocessor and logistic regression using only that fold's training rows; derive training logits with that fitted regression; fit boosting trees on training rows using these logits; generate validation/test logits using that same regression, and pass them as prediction-time `base_margin`. No validation labels enter either stage. This is sequential additive boosting with a frozen logistic offset, **not** a fitted linear + tree probability average. Validation predictions are still out-of-fold even though the booster's training offsets were estimated in-sample on the training fold.

**Model assumptions:** The logistic component has the assumptions audited in Section 5.1. The tree correction can relax functional-form restrictions and capture interactions but cannot resolve repeated-observation dependence, leakage, distribution shift, or violations of a meaningful sampling protocol. The corrected model loses the strict linear-logit interpretation of its *final* predictions, although the baseline coefficients remain interpretable as a component. We compare its ROC AUC, average precision, log loss, and Brier score with standalone models; improvement is not guaranteed.


In [ ]:
# Diagnostic PCA plot: compute from a small, random sample *only for visualization*.
plot_sample = X.sample(n=min(20_000, len(X)), random_state=RANDOM_STATE)
plot_preprocess = make_preprocessor(scale_numeric=True)
Z_plot = plot_preprocess.fit_transform(plot_sample)
pca_diagnostic = PCA(svd_solver='full').fit(Z_plot)
cumulative = np.cumsum(pca_diagnostic.explained_variance_ratio_)
k90 = int(np.searchsorted(cumulative, 0.90) + 1)
print(f'Diagnostic PCA: {k90} of {Z_plot.shape[1]} components explain {cumulative[k90-1]:.2%} of preprocessed variance')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(cumulative) + 1), cumulative, marker='.', markersize=3)
ax.axhline(0.90, color='tab:red', linestyle='--', label='90% variance')
ax.axvline(k90, color='tab:green', linestyle=':', label=f'{k90} components')
ax.set(xlabel='Number of principal components', ylabel='Cumulative explained variance', ylim=(0, 1.03))
ax.set_title('PCA diagnostic: information retained by principal components')
ax.legend(); finish_plot(fig, '16_pca_explained_variance')

## 6. Cross-validated hyperparameter experiments

Each candidate $m$ produces full-length OOF predictions $\hat p_m^{\mathrm{OOF}}$ and an averaged test prediction

$$\hat p_m^{\mathrm{test}}(x)=\frac{1}{5}\sum_{j=1}^5 \hat p_{m,j}(x).$$

Thus every hyperparameter candidate is retained. By default, the five fitted fold pipelines for each candidate are also saved (`SAVE_FOLD_MODELS=True`), potentially using several gigabytes. Set this to `False` if storage is limited; all metrics, OOF predictions, and averaged test predictions are still retained for reproducing selection and blending.

Primary metric: **ROC AUC**. Secondary metrics: **average precision (area under the precision–recall curve using stepwise interpolation), log loss, Brier score, precision, recall and F1 at 0.5, plus precision and recall in the riskiest 1%**. Precision/recall at 0.5 are descriptive; the threshold is *not* optimized here.

In [ ]:
def classification_metrics(true_y, probabilities) -> dict:
    true_y = np.asarray(true_y, dtype=int)
    p = np.clip(np.asarray(probabilities, dtype=float), 1e-7, 1.0 - 1e-7)
    predicted = (p >= 0.5).astype(int)
    k = max(1, int(np.ceil(0.01 * len(true_y))))
    top = np.argpartition(p, -k)[-k:]
    return {
        'roc_auc': float(roc_auc_score(true_y, p)),
        'average_precision': float(average_precision_score(true_y, p)),
        'log_loss': float(log_loss(true_y, p, labels=[0, 1])),
        'brier': float(brier_score_loss(true_y, p)),
        'precision_at_0p5': float(precision_score(true_y, predicted, zero_division=0)),
        'recall_at_0p5': float(recall_score(true_y, predicted, zero_division=0)),
        'f1_at_0p5': float(f1_score(true_y, predicted, zero_division=0)),
        'precision_at_top_1pct': float(true_y[top].mean()),
        'recall_at_top_1pct': float(true_y[top].sum() / max(1, true_y.sum())),
    }

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_splits = list(cv.split(X, y))
assert all(len(np.unique(y.iloc[va])) == 2 for _, va in fold_splits)
fold_assignments = np.full(len(X), -1, dtype=np.int8)
for fold, (_, valid_idx) in enumerate(fold_splits, start=1):
    fold_assignments[valid_idx] = fold
assert (fold_assignments > 0).all()
# These files depend on private competition records: keep outputs/ out of public Git.

predictions = {}
fold_records, candidate_records = [], []

def evaluate_candidate(spec: dict):
    name = spec['name']
    oof = np.full(len(X), np.nan, dtype=np.float32)
    pred_test = np.zeros(len(X_test), dtype=np.float64)
    start_candidate = time.perf_counter()
    is_offset_model = spec['family'] == 'logistic_offset_xgb'
    if is_offset_model:
        oof_base = np.full(len(X), np.nan, dtype=np.float32)
        oof_delta = np.full(len(X), np.nan, dtype=np.float32)
    for fold, (tr_idx, va_idx) in enumerate(fold_splits, start=1):
        model = make_pipeline(spec)
        start_fold = time.perf_counter()
        with warnings.catch_warnings(record=True) as fitting_warnings:
            warnings.simplefilter('always', ConvergenceWarning)
            model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        p_valid = model.predict_proba(X.iloc[va_idx])[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]
        assert np.isfinite(p_valid).all() and np.isfinite(p_test).all()
        if is_offset_model:
            valid_transformed = model.named_steps['preprocess'].transform(X.iloc[va_idx])
            base_p, corrected_p, delta_logit = model.named_steps['model'].predict_components(valid_transformed)
            assert np.allclose(p_valid, corrected_p, atol=1e-7)
            oof_base[va_idx] = base_p
            oof_delta[va_idx] = delta_logit
        oof[va_idx] = p_valid
        pred_test += p_test / N_SPLITS
        scores = classification_metrics(y.iloc[va_idx], p_valid)
        scores.update(candidate=name, family=spec['family'], fold=fold,
                      fit_predict_seconds=round(time.perf_counter() - start_fold, 2))
        if spec['family'].startswith('logistic'):
            logistic_estimator = (model.named_steps['model'].logistic_
                                  if is_offset_model else model.named_steps['model'])
            scores['logit_iterations'] = int(logistic_estimator.n_iter_[0])
            scores['convergence_warning'] = any(
                issubclass(w.category, ConvergenceWarning) for w in fitting_warnings)
        else:
            scores['logit_iterations'] = np.nan
            scores['convergence_warning'] = False
        if 'pca' in model.named_steps:
            scores['pca_components'] = int(model.named_steps['pca'].n_components_)
        fold_records.append(scores)
        if SAVE_FOLD_MODELS:
            model_path = OUTPUT_DIR / 'models' / f'{name}_fold{fold}.joblib'
            model_path.parent.mkdir(exist_ok=True, parents=True)
            joblib.dump(model, model_path, compress=3)
        print(f'{name:<25s} fold {fold}/{N_SPLITS}: AUC={scores["roc_auc"]:.5f}, AP={scores["average_precision"]:.5f}', flush=True)
        del model
    assert np.isfinite(oof).all()
    predictions[name] = {'oof': oof, 'test': pred_test}
    if is_offset_model:
        assert np.isfinite(oof_base).all() and np.isfinite(oof_delta).all()
        predictions[name]['baseline_oof'] = oof_base
        predictions[name]['correction_logit_oof'] = oof_delta
        np.savez_compressed(OUTPUT_DIR / f'{name}_components_oof.npz',
                            baseline_prob=oof_base, correction_logit=oof_delta,
                            corrected_prob=oof)
    aggregated = classification_metrics(y, oof)
    aggregated.update(candidate=name, family=spec['family'],
                      total_seconds=round(time.perf_counter() - start_candidate, 2))
    candidate_records.append(aggregated)
    # Incremental checkpoint: if the notebook is interrupted, earlier predictions survive.
    pd.DataFrame(fold_records).to_csv(OUTPUT_DIR / 'fold_metrics.csv', index=False)
    pd.DataFrame(candidate_records).to_csv(OUTPUT_DIR / 'candidate_metrics.csv', index=False)
    np.savez_compressed(OUTPUT_DIR / 'all_candidate_predictions.npz',
                        **{f'{n}__{part}': p[part] for n, p in predictions.items() for part in ('oof', 'test')})
    with (OUTPUT_DIR / 'candidate_specs.json').open('w') as file:
        json.dump(specs, file, indent=2)
    return aggregated

## 7. Run all five-fold experiments

**Full run:** 15 candidate configurations × 5 folds = 75 fitted pipelines. On 699,960 rows, this is computationally substantial and may take considerable time. The notebook does not claim full-run results until this cell finishes. If necessary, smoke-test with the settings above, then restore `FAST_DEV_RUN=False` for final results.

In [ ]:
for spec in specs:
    evaluate_candidate(spec)

leaderboard = pd.DataFrame(candidate_records).sort_values('roc_auc', ascending=False).reset_index(drop=True)
fold_scores = pd.DataFrame(fold_records)
display(leaderboard[['candidate','family','roc_auc','average_precision','log_loss','brier',
                     'precision_at_top_1pct','recall_at_top_1pct','total_seconds']].round(5))

## 8. Compare candidates visually

Compare OOF ROC curves for the best candidate of each family. All models use the **same folds**, enabling paired comparisons. AUC confidence intervals are not provided here; fold-score standard deviations are descriptive and are not formal uncertainty intervals.

In [ ]:
fold_auc_summary = fold_scores.groupby('candidate')['roc_auc'].agg(['mean','std']).sort_values('mean', ascending=False)
display(fold_auc_summary)

best_each_family = (leaderboard.sort_values('roc_auc', ascending=False)
                    .drop_duplicates('family')['candidate'].tolist())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name in best_each_family:
    fpr, tpr, _ = roc_curve(y, predictions[name]['oof'])
    axes[0].plot(fpr, tpr, label=f'{name} ({roc_auc_score(y, predictions[name]["oof"]):.4f})')
    precision, recall, _ = precision_recall_curve(y, predictions[name]['oof'])
    axes[1].plot(recall, precision, label=name)
axes[0].plot([0,1],[0,1],ls='--',color='gray')
axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='OOF ROC curves')
axes[1].axhline(y.mean(),ls='--',color='gray',label='Class prevalence')
axes[1].set(xlabel='Recall',ylabel='Precision',title='OOF precision–recall curves')
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
finish_plot(fig, '15_oof_roc_pr_curves')

### 8.1 Compare tuned models and fold stability

The *point estimates* below are five-fold OOF candidate AUCs. The fold boxplots describe heterogeneity among stratified folds and are **not confidence intervals** for an independently chosen winner. The bar chart uses the same OOF score for every model, and the next plot quantifies how similar their predictions are.

In [ ]:
plot_board = leaderboard.sort_values('roc_auc', ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(17, max(5, .42 * len(plot_board)+2)))
sns.barplot(data=plot_board, x='roc_auc', y='candidate', hue='family',
            palette='colorblind', dodge=False, ax=axes[0])
axes[0].set(title='Candidate ranking by full OOF ROC AUC', xlabel='Out-of-fold ROC AUC (higher = better)', ylabel='Candidate')
axes[0].legend(title='Model family', fontsize=8)
sns.boxplot(data=fold_scores, x='roc_auc', y='candidate', hue='family',
            palette='colorblind', dodge=False, ax=axes[1], fliersize=2)
sns.stripplot(data=fold_scores, x='roc_auc', y='candidate', color='#253858',
              size=3, jitter=.10, alpha=.7, ax=axes[1])
axes[1].set(title='Stratified-fold AUC variation', xlabel='Fold ROC AUC', ylabel='Candidate')
legend=axes[1].get_legend()
if legend is not None: legend.remove()
finish_plot(fig, '12_candidate_scores_and_folds')

### 8.2 Prediction agreement and redundancy

Highly correlated OOF probability vectors often provide less ensemble diversity. This is a descriptive **Pearson correlation among model scores**, not a proof of whether a blend will improve ROC AUC. Two models can be strongly correlated and still differ on crucial positive cases.

In [ ]:
oof_matrix = pd.DataFrame({name: predictions[name]['oof'] for name in predictions})
corr_predictions = oof_matrix.corr()
fig, ax = plt.subplots(figsize=(max(7, .72*len(oof_matrix.columns)), max(6, .65*len(oof_matrix.columns))))
sns.heatmap(corr_predictions, vmin=0, vmax=1, cmap='crest', annot=len(predictions)<=8,
            fmt='.2f', square=True, linewidths=.5, ax=ax, cbar_kws={'label':'Pearson correlation (OOF fraud scores)'})
ax.set_title('Similarity among out-of-fold probability predictions')
ax.tick_params(axis='x', labelrotation=65)
finish_plot(fig, '13_oof_prediction_correlations')

### 8.3 Out-of-fold specification, calibration, and convergence checks

Use **fixed, predeclared** candidates `logit_C1` and `logit_spline_C1`, not whichever happened to win. This diagnostic is available **only after the CV loop executes**. Comparing their OOF ROC AUC/log loss gives evidence about the **predictive value of nonlinear log-odds**, not a formal test that the linear model's functional-form assumption holds. These OOF comparisons remain exploratory because the feature set was informed by full-training-set EDA.

For a well-calibrated model, bins of predictions with mean score $\bar p_b$ should have event frequency $\bar y_b\approx\bar p_b$. The **reliability diagram** displays both quantities in quantile bins. Calibration is important when scores are interpreted as probabilities; Kaggle's ROC AUC evaluates ranking instead.

The residual plot uses **binned means of $Y-\hat p$**, rather than treating Bernoulli residuals like Gaussian residuals. Deviations may signal missing nonlinear effects or interactions, but do not by themselves establish significance. No arbitrary 0.5 threshold enters these checks.

In [ ]:
# Check a predetermined pair, not the post-selection winner.
comparison_names = [name for name in ['logit_C1', 'logit_spline_C1'] if name in predictions]
if comparison_names:
    compare = leaderboard.set_index('candidate').loc[comparison_names,
                   ['roc_auc','average_precision','log_loss','brier']]
    display(compare)
    if len(comparison_names) == 2:
        print('Spline minus ordinary logistic (OOF):',
              {k:round(float(compare.loc['logit_spline_C1', k] - compare.loc['logit_C1', k]), 6)
               for k in compare.columns})

    # Paired fold convergence audits (no warnings is NOT proof of a correct model).
    warning_audit = fold_scores[fold_scores.candidate.isin(comparison_names)][
        ['candidate','fold','logit_iterations','convergence_warning','roc_auc','log_loss']]
    display(warning_audit)
    warning_audit.to_csv(OUTPUT_DIR/'assumptions_logit_fold_convergence.csv', index=False)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    calibration_rows = []
    for name in comparison_names:
        oof = np.asarray(predictions[name]['oof'], dtype=float)
        # Quantile bins avoid mostly empty bins in this very imbalanced task.
        bins = pd.qcut(oof, q=10, duplicates='drop')
        checks = pd.DataFrame({'bin':bins,'observed':y.to_numpy(), 'predicted':oof})\
            .groupby('bin', observed=True).agg(events=('observed','sum'),
               n=('observed','size'), observed_rate=('observed','mean'),
               predicted_rate=('predicted','mean'))
        calibration_rows += [dict(candidate=name, bin=str(i), **row.to_dict())
                             for i,row in checks.iterrows()]
        axes[0].plot(checks.predicted_rate, checks.observed_rate,
                     marker='o', label=name)
    axes[0].plot([0,1],[0,1],linestyle='--',color='gray',label='Perfect calibration')
    axes[0].set(title='Out-of-fold reliability (10 equal-count score bins)',
                xlabel='Mean predicted probability',ylabel='Observed fraud frequency',
                xlim=(0, .4), ylim=(0, .4))
    axes[0].legend()
    calibration_df = pd.DataFrame(calibration_rows)
    calibration_df.to_csv(OUTPUT_DIR/'assumptions_oof_calibration_bins.csv',index=False)

    # Standard logistic residual pattern by a high-interest numeric feature.
    residual_feature = 'merchant_risk_score'
    if residual_feature in X.columns and 'logit_C1' in predictions:
        residuals = y.to_numpy() - np.asarray(predictions['logit_C1']['oof'],dtype=float)
        residual_frame = pd.DataFrame({'x':X[residual_feature].to_numpy(),'residual':residuals})
        residual_frame['bin'] = pd.qcut(residual_frame.x, q=15, duplicates='drop')
        res = residual_frame.groupby('bin',observed=True).agg(
            mean_x=('x','mean'),mean_residual=('residual','mean'),count=('residual','size'))
        sns.lineplot(data=res,x='mean_x',y='mean_residual',marker='o',ax=axes[1],
                     label='Standard logistic: observed − predicted')
        res.to_csv(OUTPUT_DIR/'assumptions_oof_residual_bins.csv')
    axes[1].axhline(0,color='#E45756',linestyle='--',label='Zero mean residual')
    axes[1].set(title='OOF binned probability residuals versus merchant risk',
                xlabel='Mean merchant risk score in bin',ylabel='Observed fraud − predicted fraud rate')
    axes[1].legend()
    finish_plot(fig, '18_oof_logistic_assumption_checks')
else:
    print('No fixed logistic candidate OOF predictions available: rerun Section 7.')

### 8.4 Does residual boosting improve on logistic regression?

Our predefined comparison is **`logit_C1` versus `offset_xgb_depth2_lr05`**, and, if available, `offset_xgb_depth3_lr05` and standalone `xgb_depth3_lr08`. All use *identical outer CV folds*. The boosted model's baseline is the logistic model fitted **inside each fold**. The visualizations show (1) binned OOF residuals before and after correction, (2) how correction magnitude depends on the logistic baseline risk, and (3) changes in ROC AUC and average precision. The reference line represents zero mean error, **not** a test that every observation is predictable.

The booster optimizes Bernoulli log loss, but Kaggle scores ROC AUC; therefore a reduction in residual patterns or log loss need not improve AUC. AUC differences after choosing hyperparameters on the same OOF sample are *exploratory*. A nested validation or separate holdout is needed to estimate the selected winner's generalization advantage.


In [ ]:
fixed_comparison = [name for name in
    ['logit_C1', 'offset_xgb_depth2_lr05', 'offset_xgb_depth3_lr05', 'xgb_depth3_lr08']
    if name in predictions]
if 'logit_C1' in fixed_comparison and any('offset_xgb' in n for n in fixed_comparison):
    display(leaderboard.set_index('candidate').loc[fixed_comparison,
                 ['roc_auc', 'average_precision', 'log_loss', 'brier']].round(6))
    comparison_fold = fold_scores[fold_scores.candidate.isin(fixed_comparison)][
        ['candidate', 'fold', 'roc_auc', 'average_precision', 'log_loss', 'brier']]
    comparison_fold.to_csv(OUTPUT_DIR / 'offset_boost_fold_comparison.csv', index=False)
    # Use fixed, predeclared candidates to reduce post-selection bias in the illustration.
    chosen_offset = next(n for n in fixed_comparison if 'offset_xgb' in n)
    base_oof = np.asarray(predictions[chosen_offset]['baseline_oof'])
    boost_oof = np.asarray(predictions[chosen_offset]['oof'])
    y_array = y.to_numpy()
    feature_to_plot = 'merchant_risk_score'
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
    if feature_to_plot in X.columns:
        frame = pd.DataFrame({'feature': X[feature_to_plot].to_numpy(),
                             'baseline_residual': y_array - base_oof,
                             'corrected_residual': y_array - boost_oof})
        frame['bin'] = pd.qcut(frame['feature'], q=15, duplicates='drop')
        grouped = frame.groupby('bin', observed=True).agg(
            feature=('feature', 'mean'), base=('baseline_residual', 'mean'),
            corrected=('corrected_residual', 'mean'), count=('feature', 'size'))
        residual_lines = grouped.reset_index().melt(id_vars=['feature'],
            value_vars=['base', 'corrected'], var_name='stage', value_name='mean_residual')
        sns.lineplot(data=residual_lines, x='feature', y='mean_residual', hue='stage',
                     style='stage', markers=True, dashes=False, ax=axes[0])
        grouped.to_csv(OUTPUT_DIR / 'offset_boost_binned_residuals.csv', index=False)
    axes[0].axhline(0, linestyle='--', color='gray', label='Zero mean residual')
    axes[0].set(title='Out-of-fold residuals before vs after boosting',
                xlabel='Merchant risk score (bin mean)',
                ylabel='Mean observed − predicted fraud probability')
    axes[0].legend(title='Stage')

    # Quantile-binned correction on logit scale, based only on held-out OOF predictions.
    correction_frame = pd.DataFrame({'baseline_probability':base_oof,
                      'delta_logit':predictions[chosen_offset]['correction_logit_oof']})
    correction_frame['bin'] = pd.qcut(correction_frame['baseline_probability'],
                                      q=20, duplicates='drop')
    corrections = correction_frame.groupby('bin', observed=True).agg(
        baseline_p=('baseline_probability', 'mean'),
        mean_logit_correction=('delta_logit', 'mean'),
        n=('delta_logit', 'size'))
    sns.lineplot(data=corrections, x='baseline_p', y='mean_logit_correction',
                 marker='o', ax=axes[1], label='Average boosted correction')
    axes[1].axhline(0, linestyle='--', color='gray', label='No correction')
    axes[1].set(title='Boosted correction versus logistic baseline risk',
                xlabel='Mean logistic predicted fraud probability',
                ylabel='Mean added log-odds')
    axes[1].legend()
    corrections.to_csv(OUTPUT_DIR / 'offset_boost_logit_corrections.csv')

    # Paired candidate-level AUC/AP comparison; values from identical CV held-out rows.
    compare_metrics = leaderboard.set_index('candidate').loc[fixed_comparison,
                                    ['roc_auc','average_precision']].reset_index()
    long_metrics = compare_metrics.melt(id_vars='candidate', var_name='metric', value_name='score')
    sns.barplot(data=long_metrics, x='candidate', y='score', hue='metric', ax=axes[2])
    axes[2].set(title='Out-of-fold quality: baseline, correction, standalone',
                xlabel='Prespecified model', ylabel='OOF score (higher is better)')
    axes[2].tick_params(axis='x', labelrotation=40)
    axes[2].legend(title='Metric')
    finish_plot(fig, '19_logistic_offset_boosting_diagnostics')
else:
    print('Offset boosting model or fixed logistic baseline was not evaluated.')


### 8.5 Joint model quality: ranking, probability quality, and compute

ROC AUC ranks fraud cases against legitimate cases, average precision measures precision across recall thresholds, log loss penalizes overconfident wrong probabilities, and Brier score measures squared probability error. These metrics need not select the same model. Training time also matters, but wall-clock measurements depend on hardware and concurrent workload. All displayed quality metrics come from OOF predictions.

In [ ]:
quality_metrics=['roc_auc','average_precision','log_loss','brier']
quality_display=leaderboard.melt(id_vars=['candidate','family'],value_vars=quality_metrics,
                                var_name='metric',value_name='score')
fig,axs=plt.subplots(2,2,figsize=(17,12))
for ax,metric in zip(axs.flat,quality_metrics):
    sub=quality_display.query('metric == @metric').sort_values('score',ascending=(metric in ['log_loss','brier']))
    sns.barplot(data=sub,x='score',y='candidate',hue='family',dodge=False,ax=ax)
    ax.set(title=f'{metric.replace("_"," ").title()} | {"lower" if metric in ["log_loss","brier"] else "higher"} is better',
           xlabel=metric.replace('_',' ').title(),ylabel='Candidate')
    lg=ax.get_legend()
    if lg:lg.remove()
fig.suptitle('OOF model comparison across four distinct quality criteria',y=1.01)
finish_plot(fig,'26_four_metric_model_scorecard')
fig,ax=plt.subplots(figsize=(12,6))
sns.scatterplot(data=leaderboard,x='total_seconds',y='roc_auc',hue='family',size='average_precision',
                sizes=(50,240),alpha=.85,ax=ax)
for _,row in leaderboard.iterrows():
    ax.annotate(row['candidate'],(row['total_seconds'],row['roc_auc']),xytext=(3,3),
                textcoords='offset points',fontsize=7,alpha=.85)
ax.set(title='OOF ROC AUC versus training cost across all five folds',
       xlabel='Total fit + prediction time (seconds, platform dependent)',ylabel='OOF ROC AUC')
ax.legend(title='Model family / average precision',bbox_to_anchor=(1.02,1),loc='upper left',fontsize=8)
finish_plot(fig,'27_performance_vs_training_time')

### 8.6 Out-of-fold score distributions by actual class

A good ranking model tends to assign larger scores to fraudulent transactions, but strong overlap is inevitable. We display **within-class normalized** distributions so the rare class remains visible. This is not the population distribution of predicted fraud probabilities.

In [ ]:
family_representatives=best_each_family[:5]
fig,axs=plt.subplots(len(family_representatives),1,figsize=(13,max(7,2.6*len(family_representatives))),sharex=True)
axs=np.atleast_1d(axs)
score_sample_idx=np.random.default_rng(RANDOM_STATE).choice(len(y),size=min(25000,len(y)),replace=False)
for ax,name in zip(axs,family_representatives):
    frame=pd.DataFrame({'score':np.asarray(predictions[name]['oof'])[score_sample_idx],
                        'label':y.iloc[score_sample_idx].to_numpy()})
    frame['score_log10']=np.log10(np.clip(frame['score'],1e-6,1))
    sns.histplot(data=frame,x='score_log10',hue='label',hue_order=[0,1],palette=['#4C78A8','#E45756'],
                 bins=45,stat='density',common_norm=False,element='step',fill=False,ax=ax)
    ax.set(title=name, xlabel='log10(predicted fraud probability)',ylabel='Within-class density')
    lg=ax.get_legend()
    if lg:lg.set_title('True class (0 legitimate, 1 fraud)')
fig.suptitle('Held-out (OOF) score distributions by observed class | sampled rows',y=1.01)
finish_plot(fig,'28_oof_score_distributions')

### 8.7 Cumulative gains and lift by review budget

For score-sorted transactions, **cumulative capture** (recall) at budget $k$ is the fraction of all fraudulent transactions among the highest-scoring $k$ fraction of rows. **Lift** is cumulative precision divided by the base fraud prevalence. These are useful for a hypothetical limited review capacity, not a recommendation of a particular real-world budget.

In [ ]:
review_fracs=np.array([.001,.002,.005,.01,.02,.03,.05,.075,.10,.15,.20,.30,.50,1.0])
review_rows=[]
label_array=y.to_numpy(dtype=int)
for name in family_representatives:
    ordering=np.argsort(-np.asarray(predictions[name]['oof']),kind='stable')
    positives=np.cumsum(label_array[ordering]);total_pos=label_array.sum()
    for frac in review_fracs:
        k=min(len(y),max(1,int(np.ceil(frac*len(y)))))
        found=int(positives[k-1]);recall=found/total_pos;precision=found/k
        review_rows.append({'candidate':name,'review_fraction':frac,'review_pct':frac*100,
                            'n_reviewed':k,'frauds_captured':found,'recall':recall,
                            'precision':precision,'lift':precision/float(y.mean())})
review_budget=pd.DataFrame(review_rows)
review_budget.to_csv(OUTPUT_DIR/'model_review_budget_curves.csv',index=False)
fig,axs=plt.subplots(1,2,figsize=(17,6))
sns.lineplot(data=review_budget,x='review_pct',y='recall',hue='candidate',marker='o',ax=axs[0])
axs[0].plot([0,100],[0,1],ls='--',color='gray',label='Random ranking')
axs[0].set(title='Cumulative fraud capture versus transaction review budget',
           xlabel='Highest-risk transactions reviewed (%)',ylabel='Fraction of all fraudulent transactions captured',
           xlim=(0,50),ylim=(0,1))
axs[0].legend(title='OOF model',fontsize=8)
sns.lineplot(data=review_budget,x='review_pct',y='lift',hue='candidate',marker='o',ax=axs[1])
axs[1].axhline(1,ls='--',color='gray',label='Random ranking (lift = 1)')
axs[1].set(title='Cumulative precision lift versus review budget',
           xlabel='Highest-risk transactions reviewed (%)',ylabel='Cumulative precision / overall prevalence',
           xlim=(0,50))
axs[1].legend(title='OOF model',fontsize=8)
finish_plot(fig,'29_cumulative_gains_and_lift')

### 8.8 Precision–recall at fixed review capacity

At a fixed top-$k$ budget, model thresholds are derived by sorting **OOF** scores, not assuming a probability threshold of 0.5. Precision and recall form a trade-off; apparent optimal budget selection on these OOF curves would introduce selection bias.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(16,6))
small_budget=review_budget[review_budget['review_pct']<=10]
sns.lineplot(data=small_budget,x='review_pct',y='precision',hue='candidate',marker='o',ax=axs[0])
sns.lineplot(data=small_budget,x='review_pct',y='recall',hue='candidate',marker='o',ax=axs[1])
axs[0].axhline(float(y.mean()),ls='--',color='gray',label='Overall fraud prevalence')
for ax,metric in zip(axs,['Precision','Recall']):
    ax.set(title=f'{metric} among top-ranked transactions | OOF',
           xlabel='Transactions reviewed (%)',ylabel=metric,xlim=(0,10),ylim=(0,1))
    ax.legend(title='OOF model',fontsize=8)
finish_plot(fig,'30_precision_recall_at_review_budget')

### 8.9 Confusion matrices: fixed 0.5 threshold versus top 1%

A confusion matrix depends on a **decision rule**; it is not a threshold-free measure like ROC AUC. The left column uses an illustrative probability threshold of 0.5, while the right column flags exactly the 1% highest-scoring observations. Both are descriptive OOF evaluations. In this imbalanced dataset, 0.5 is not necessarily appropriate for fraud investigations.

In [ ]:
shown_names=list(dict.fromkeys([name for name in ['logit_C1',leaderboard.iloc[0]['candidate']] if name in predictions]))
fig,axs=plt.subplots(len(shown_names),2,figsize=(13,5*len(shown_names)),squeeze=False)
cm_rows=[]
for row,name in enumerate(shown_names):
    scores=np.asarray(predictions[name]['oof'])
    thresholded=(scores>=.5).astype(int)
    topk=np.zeros(len(scores),dtype=int)
    topk[np.argsort(-scores,kind='stable')[:max(1,int(np.ceil(.01*len(y))))]]=1
    for col,(rule,guess) in enumerate([('Threshold ≥ 0.5',thresholded),('Top 1% risk',topk)]):
        cm=confusion_matrix(y,guess,labels=[0,1]);tn,fp,fn,tp=cm.ravel()
        labels=np.array([[f'TN\n{tn:,}',f'FP\n{fp:,}'],[f'FN\n{fn:,}',f'TP\n{tp:,}']])
        sns.heatmap(cm,annot=labels,fmt='',cmap='Blues',ax=axs[row,col],cbar=False,
                    xticklabels=['Predicted 0','Predicted 1'],yticklabels=['Actual 0','Actual 1'])
        axs[row,col].set(title=f'{name} | {rule}',xlabel='Predicted class',ylabel='Observed class')
        cm_rows.append({'candidate':name,'rule':rule,'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp),
                        'recall':float(tp/(tp+fn)),'precision':float(tp/max(1,tp+fp))})
fig.suptitle('Decision errors under two prespecified operating rules (OOF)',y=1.01)
finish_plot(fig,'31_confusion_matrices_by_policy')
pd.DataFrame(cm_rows).to_csv(OUTPUT_DIR/'model_confusion_matrices.csv',index=False)

### 8.10 Calibration with uncertainty: is 10% predicted risk approximately 10% observed fraud?

Predicted probabilities need not be calibrated merely because ROC AUC is high. We divide each model's OOF scores into quantile bins and display the mean score, observed event rate, sample size and **Wilson 95% binomial interval**. These intervals describe empirical binomial uncertainty only; overlapping CV training sets and adaptive model choice make them **not formal independent confidence intervals for overall model performance**.

In [ ]:
cal_names=list(dict.fromkeys([n for n in ['logit_C1']+family_representatives if n in predictions]))[:6]
cal_rows=[]
for name in cal_names:
    p_hat=np.asarray(predictions[name]['oof'],dtype=float)
    frame=pd.DataFrame({'score':p_hat,'label':y.to_numpy()})
    frame['bin']=pd.qcut(frame.score,q=20,duplicates='drop')
    summary=frame.groupby('bin',observed=True).agg(mean_p=('score','mean'),n=('label','size'),events=('label','sum'))
    summary['observed_rate']=summary.events/summary.n
    summary['low'],summary['high']=wilson_bounds(summary.events.to_numpy(),summary.n.to_numpy())
    cal_rows.extend([dict(candidate=name,bin=str(idx),**r.to_dict()) for idx,r in summary.iterrows()])
cal_table=pd.DataFrame(cal_rows)
cal_table.to_csv(OUTPUT_DIR/'model_reliability_20bins.csv',index=False)
fig,ax=plt.subplots(figsize=(10,8))
for name in cal_names:
    s=cal_table[cal_table.candidate==name].sort_values('mean_p')
    ax.plot(s.mean_p,s.observed_rate,marker='o',ms=3,lw=1.8,label=name)
ax.plot([1e-5,1],[1e-5,1],ls='--',color='gray',label='Perfect calibration')
ax.set(xscale='log',yscale='log',xlim=(1e-5,1),ylim=(1e-5,1),
       title='OOF reliability by model | 20 equal-count bins (log–log axes)',
       xlabel='Mean predicted fraud probability',ylabel='Observed fraud frequency')
ax.legend(title='OOF candidate',fontsize=9)
finish_plot(fig,'32_model_reliability_comparison')
# A second plot exposes Wilson interval widths without overplotting six models.
focus=cal_table[cal_table.candidate==cal_names[0]].sort_values('mean_p')
fig,ax=plt.subplots(figsize=(10,5))
ax.errorbar(focus.mean_p,focus.observed_rate,
            yerr=[np.maximum(0,focus.observed_rate-focus.low),
                  np.maximum(0,focus.high-focus.observed_rate)],
            fmt='o-',capsize=3,label=f'{cal_names[0]} observed ± Wilson 95%')
max_axis=max(.001,float(max(focus.mean_p.max(),focus.observed_rate.max())*1.12))
ax.plot([0,max_axis],[0,max_axis],ls='--',color='gray',label='Perfect calibration')
ax.set(xlim=(0,max_axis),ylim=(0,max_axis),title=f'Calibration with bin uncertainty: {cal_names[0]}',
       xlabel='Mean predicted probability',ylabel='Observed fraud frequency')
ax.legend()
finish_plot(fig,'33_calibration_with_wilson_intervals')

### 8.11 Error analysis across transaction subgroups

A high global AUC can hide heterogeneity. Compute subgroup AUC and average precision on OOF scores for the prespecified logistic baseline and the OOF-leading candidate. We require at least **20 fraud cases and 20 legitimate cases per subgroup** to avoid undefined/unstable comparisons. These are exploratory subgroup estimates (not group fairness guarantees).

In [ ]:
slice_features=[c for c in ['channel','merchant_category','auth_method','device_type'] if c in X.columns]
slice_names=list(dict.fromkeys([n for n in ['logit_C1',leaderboard.iloc[0]['candidate']] if n in predictions]))
slice_rows=[]
for f in slice_features:
    for group_value,ids in X.groupby(f,observed=True).indices.items():
        yi=y.iloc[ids].to_numpy(); nfraud=int(yi.sum()); nnon=len(yi)-nfraud
        if nfraud<20 or nnon<20:continue
        for name in slice_names:
            s=np.asarray(predictions[name]['oof'])[ids]
            slice_rows.append({'feature':f,'group':str(group_value),'candidate':name,'n':len(ids),
                               'frauds':nfraud,'prevalence':nfraud/len(ids),
                               'roc_auc':roc_auc_score(yi,s),'average_precision':average_precision_score(yi,s)})
slice_scores=pd.DataFrame(slice_rows)
slice_scores.to_csv(OUTPUT_DIR/'model_subgroup_oof_metrics.csv',index=False)
if not slice_scores.empty:
    for feature in slice_features[:2]:
        this=slice_scores[slice_scores.feature==feature]
        fig,axs=plt.subplots(1,2,figsize=(16,max(4,.46*this.group.nunique()+2)))
        sns.barplot(data=this,x='roc_auc',y='group',hue='candidate',ax=axs[0])
        sns.barplot(data=this,x='average_precision',y='group',hue='candidate',ax=axs[1])
        axs[0].set(title=f'OOF ROC AUC by {feature}',xlabel='Subgroup ROC AUC',ylabel=feature)
        axs[1].set(title=f'OOF average precision by {feature}',xlabel='Subgroup average precision',ylabel=feature)
        for ax in axs:ax.legend(title='Candidate',fontsize=8)
        fig.suptitle(f'Observed subgroup ranking quality | {feature} (≥20 cases per class)',y=1.01)
        finish_plot(fig,f'34_subgroups_{feature}')

### 8.12 Do models disagree on the riskiest transactions?

Prediction correlation was plotted earlier. This heatmap instead compares each pair's **top-1%-of-transactions sets**, using Jaccard overlap $|A\cap B|/|A\cup B|$. Low overlap indicates different review candidates, but does not guarantee better ensembles; higher overlap may simply reflect shared signal.

In [ ]:
selected_for_overlap=best_each_family[:6]
selection_sets={name:set(np.argsort(-np.asarray(predictions[name]['oof']),kind='stable')[:max(1,int(np.ceil(.01*len(y))))]) for name in selected_for_overlap}
overlap=pd.DataFrame(index=selected_for_overlap,columns=selected_for_overlap,dtype=float)
for a in selected_for_overlap:
    for b in selected_for_overlap:
        overlap.loc[a,b]=len(selection_sets[a]&selection_sets[b])/len(selection_sets[a]|selection_sets[b])
overlap.to_csv(OUTPUT_DIR/'model_top1pct_jaccard.csv')
fig,ax=plt.subplots(figsize=(max(7,1.25*len(overlap)),max(6,1.15*len(overlap))))
sns.heatmap(overlap,annot=True,fmt='.2f',cmap='YlGnBu',vmin=0,vmax=1,square=True,
            linewidths=.5,cbar_kws={'label':'Jaccard overlap of top 1% risk sets'},ax=ax)
ax.set(title='OOF overlap among model-defined highest-risk transaction sets',xlabel='Candidate',ylabel='Candidate')
ax.tick_params(axis='x',rotation=50)
finish_plot(fig,'35_top_risk_set_overlap')

### 8.13 Explainability: logistic coefficients and tree feature importance

Logistic coefficients describe direction/size on **transformed feature scales** (standardized numeric features, reference-dropped one-hot indicators); they should not be naively compared as causal effects. Tree `feature_importances_` are **model-internal split/gain summaries** and can overvalue correlated or high-cardinality features. They are not out-of-fold permutation importance. We inspect one saved fold without refitting; when model saving is disabled the plot is skipped.

In [ ]:
importance_names=[n for n in ['logit_C1','xgb_depth3_lr08','rf_depth12_leaf10'] if n in predictions]
importance_records=[]
if SAVE_FOLD_MODELS:
    fig,axs=plt.subplots(1,len(importance_names),figsize=(6*len(importance_names),7),squeeze=False)
    for ax,name in zip(axs.flat,importance_names):
        path=OUTPUT_DIR/'models'/f'{name}_fold1.joblib'
        if not path.exists():
            ax.text(.5,.5,f'Missing model: {path.name}',ha='center',va='center',transform=ax.transAxes)
            continue
        model=joblib.load(path)
        estimator=model.named_steps['model']
        names=np.asarray(model.named_steps['preprocess'].get_feature_names_out(),dtype=str)
        if hasattr(estimator,'coef_'):
            importance=np.asarray(estimator.coef_).ravel();metric='Signed logistic coefficient'
            order=np.argsort(np.abs(importance))[-14:]
        elif hasattr(estimator,'feature_importances_'):
            importance=np.asarray(estimator.feature_importances_).ravel();metric='Model-internal importance'
            order=np.argsort(importance)[-14:]
        else:
            ax.set_visible(False);continue
        assert len(names)==len(importance),(name,len(names),len(importance))
        small=pd.DataFrame({'feature':names[order],'value':importance[order]})
        sns.barplot(data=small,x='value',y='feature',color='#4C78A8',ax=ax)
        if hasattr(estimator,'coef_'):ax.axvline(0,color='gray',ls='--')
        ax.set(title=f'{name} | fold 1 (descriptive)',xlabel=metric,ylabel='Transformed feature')
        importance_records.extend([{'candidate':name,'fold':1,'feature':f,'importance':float(v),'type':metric} for f,v in zip(names,importance)])
        del model
    fig.suptitle('Interpreting fitted models: one training fold, original CV splits',y=1.01)
    finish_plot(fig,'36_model_feature_importance')
else: print('Feature-importance plots skipped because SAVE_FOLD_MODELS=False.')
if importance_records:pd.DataFrame(importance_records).to_csv(OUTPUT_DIR/'model_fold1_feature_importance.csv',index=False)

### 8.14 Partial dependence: comparing fitted nonlinear responses

A **partial dependence (PD)** curve averages a model's predictions when a chosen feature is set to a grid of values while other features retain their observed distribution. It is a *model-based descriptive projection*, not a causal effect. Because independently changing features can generate unrealistic combinations when predictors are correlated, interpret PD cautiously. This plot uses up to 400 randomly selected training rows and saved fold-1 models, with no additional fitting. Set `RUN_PDP=False` if evaluation is constrained by time or memory.

In [ ]:
pdp_names=[n for n in ['logit_C1','logit_spline_C1','xgb_depth3_lr08'] if n in predictions]
pdp_features=[c for c in ['merchant_risk_score','velocity_score'] if c in X.columns]
if RUN_PDP and SAVE_FOLD_MODELS and pdp_names and pdp_features:
    pdp_view=X.sample(n=min(400,len(X)),random_state=RANDOM_STATE).copy()
    fig,axs=plt.subplots(1,len(pdp_features),figsize=(15,5.5),squeeze=False)
    pdp_rows=[]
    for name in pdp_names:
        path=OUTPUT_DIR/'models'/f'{name}_fold1.joblib'
        if not path.exists():continue
        mdl=joblib.load(path)
        for feat,ax in zip(pdp_features,axs.flat):
            # Method 'brute' predicts actual probabilities through the full pipeline.
            result=partial_dependence(mdl,pdp_view,[feat],method='brute',
                                      response_method='predict_proba',grid_resolution=15,
                                      percentiles=(.05,.95))
            grid=np.asarray(result['grid_values'][0]);mean=np.asarray(result['average']).reshape(-1)
            ax.plot(grid,mean,marker='o',ms=3,lw=1.7,label=name)
            pdp_rows.extend({'candidate':name,'feature':feat,'feature_value':float(a),'mean_predicted_probability':float(b)} for a,b in zip(grid,mean))
        del mdl
    for feat,ax in zip(pdp_features,axs.flat):
        ax.set(title=f'Model response to {feat} (partial dependence)',
               xlabel=f'{feat} (sampled 5–95% grid)',ylabel='Mean model predicted fraud probability')
        ax.legend(title='Fold-1 fitted model',fontsize=9)
    fig.suptitle('Partial-dependence comparison | fixed-seed 400-row reference sample',y=1.01)
    finish_plot(fig,'37_partial_dependence_models')
    if pdp_rows:pd.DataFrame(pdp_rows).to_csv(OUTPUT_DIR/'model_partial_dependence.csv',index=False)
else: print('Partial dependence skipped (RUN_PDP=False or saved models unavailable).')

### 8.15 Hyperparameter effects and fold-by-fold stability

A primary research objective is **hyperparameter tuning**. This visualization records the actual hyperparameters for each candidate and plots their OOF ROC AUC, rather than reporting only the winner. Hyperparameters change jointly in some grid entries, so these are *controlled comparisons only when the other settings remain equal*. The fold heatmap also shows whether a model's score depends strongly on a particular validation fold. It is not a confidence interval.

In [ ]:
spec_lookup={s['name']:s for s in specs}
param_rows=[]
for _,r in leaderboard.iterrows():
    sp=spec_lookup[r['candidate']]
    params=sp['params']
    if sp['family'].startswith('logistic') and sp['family'] != 'logistic_offset_xgb':
        x_parameter=f"C={params.get('C')}" + (' / balanced' if params.get('class_weight') else '')
    elif sp['family']=='random_forest':
        x_parameter=f"depth={params.get('max_depth')}, leaf={params.get('min_samples_leaf')}"
    elif sp['family']=='xgboost':
        x_parameter=f"depth={params.get('max_depth')}, lr={params.get('learning_rate')}"
    else:
        x_parameter=f"depth={params.get('max_depth')}, lr={params.get('learning_rate')}"
    param_rows.append({'candidate':r['candidate'],'family':sp['family'],
                       'hyperparameters':x_parameter,'roc_auc':r['roc_auc'],
                       'average_precision':r['average_precision']})
param_table=pd.DataFrame(param_rows)
param_table.to_csv(OUTPUT_DIR/'model_hyperparameter_comparison.csv',index=False)
fig,axs=plt.subplots(2,3,figsize=(19,11))
for ax,(family,sub) in zip(axs.flat,param_table.groupby('family',sort=False)):
    sub=sub.sort_values('roc_auc',ascending=True)
    sns.barplot(data=sub,x='roc_auc',y='hyperparameters',color='#4C78A8',ax=ax)
    ax.set(title=f'{family}: hyperparameter choices',xlabel='OOF ROC AUC',ylabel='Tuning configuration')
for ax in axs.flat[param_table.family.nunique():]:ax.set_visible(False)
fig.suptitle('Tuning diagnostics: preserve and compare every candidate (OOF, not independent holdout)',y=1.01)
finish_plot(fig,'40_hyperparameter_effects')
fold_pivot=fold_scores.pivot(index='candidate',columns='fold',values='roc_auc')
fig,ax=plt.subplots(figsize=(11,max(6,.45*len(fold_pivot)+2)))
sns.heatmap(fold_pivot,annot=True,fmt='.3f',cmap='crest',linewidths=.5,
            cbar_kws={'label':'Held-out fold ROC AUC'},ax=ax)
ax.set(title='Fold-level model stability using identical stratified folds',
       xlabel='Held-out fold',ylabel='Candidate configuration')
finish_plot(fig,'41_fold_auc_heatmap')

## 9. Blending: keep all candidates, do not automatically average everything

Model averaging across heterogeneous families is more accurately called a **blend** than *bagging*. For a set of candidates $S$ with weights $w_m\ge 0$ and $\sum_{m\in S}w_m=1$,

$$\hat p_{\mathrm{blend}}(x)=\sum_{m\in S} w_m\hat p_m(x).$$

We compare **predefined equal-weight strategies**: the best single candidate, all candidates, the top three candidates, and one best candidate per family. All constituent models are retained in `all_candidate_predictions.npz`. Selection of the winning blend using the same OOF labels is exploratory and introduces selection optimism; report this honestly in the written project.

In [ ]:
ranked = leaderboard['candidate'].tolist()
strategies = {
    'best_single': ranked[:1],
    'all_candidates': ranked,
    'top_3': ranked[:min(3, len(ranked))],
    'best_per_family': best_each_family,
}
blend_predictions, blend_records = {}, []
for strategy, names in strategies.items():
    prob_oof = np.mean([predictions[n]['oof'] for n in names], axis=0)
    prob_test = np.mean([predictions[n]['test'] for n in names], axis=0)
    blend_predictions[strategy] = {'oof':prob_oof, 'test':prob_test}
    scores = classification_metrics(y, prob_oof)
    scores.update(strategy=strategy, n_candidates=len(names), members=', '.join(names))
    blend_records.append(scores)

blend_table = pd.DataFrame(blend_records).sort_values('roc_auc', ascending=False).reset_index(drop=True)
display(blend_table[['strategy','n_candidates','roc_auc','average_precision',
                     'log_loss','brier','precision_at_top_1pct','recall_at_top_1pct','members']])
blend_table.to_csv(OUTPUT_DIR / 'ensemble_metrics.csv', index=False)
with (OUTPUT_DIR / 'ensemble_members.json').open('w') as file:
    json.dump(strategies, file, indent=2)

# This is OOF-based model selection, not a fully independent evaluation.
chosen = blend_table.iloc[0]['strategy']
print('OOF-selected strategy:', chosen)

### 9.1 Did blending actually help?

Compare equal-weight blending strategies with the best single model. A blend that loses AUC should **not** be selected merely because it contains more trained models. Its membership and scores remain preserved for reproducibility. Note that selecting the highest OOF score among candidates and blends produces an optimistically selected estimate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ordered_blends = blend_table.sort_values('roc_auc', ascending=False)
sns.barplot(data=ordered_blends, x='roc_auc', y='strategy', color='#4C78A8', ax=axes[0])
axes[0].set(title='OOF ROC AUC by blending strategy', xlabel='ROC AUC', ylabel='Ensemble strategy')
axes[0].axvline(ordered_blends.loc[ordered_blends.strategy=='best_single','roc_auc'].iloc[0],
                color='#E45756', linestyle='--', label='Best single AUC')
axes[0].legend()
sns.barplot(data=ordered_blends, x='average_precision', y='strategy', color='#E45756', ax=axes[1])
axes[1].set(title='OOF average precision by strategy', xlabel='Average precision (higher = better)', ylabel='Ensemble strategy')
fig.suptitle('Ensemble comparison: pooled out-of-fold results', y=1.02)
finish_plot(fig, '14_ensemble_comparisons')

### 9.2 Does the OOF-selected ensemble change the risk ranking?

Compare the best individual model and the selected blend on **held-out training predictions**. A low absolute probability shift does not guarantee the ranking is unchanged, and an improved OOF score after choosing among blends is subject to selection optimism. The class-conditional distributions below show which observations receive larger or smaller estimated fraud probability in the blend.

In [ ]:
baseline_name=ranked[0]
base_scores=np.asarray(predictions[baseline_name]['oof'])
blend_scores=np.asarray(blend_predictions[chosen]['oof'])
change=blend_scores-base_scores
rng=np.random.default_rng(RANDOM_STATE)
plot_idx=rng.choice(len(change),size=min(20000,len(change)),replace=False)
frame=pd.DataFrame({'individual':base_scores[plot_idx], 'ensemble':blend_scores[plot_idx],
                    'delta':change[plot_idx],'class':y.iloc[plot_idx].to_numpy()})
fig,axs=plt.subplots(1,2,figsize=(16,6))
sns.scatterplot(data=frame,x='individual',y='ensemble',hue='class',hue_order=[0,1],
                palette=['#4C78A8','#E45756'],alpha=.24,s=11,rasterized=True,ax=axs[0])
axs[0].plot([0,1],[0,1],ls='--',color='gray',label='No change')
axs[0].set(title='Selected blend versus best individual | sample of OOF scores',
           xlabel=f'Best single: {baseline_name}',ylabel=f'OOF selected blend: {chosen}',xlim=(0,1),ylim=(0,1))
axs[0].legend(title='Actual class',fontsize=9)
sns.histplot(data=frame,x='delta',hue='class',hue_order=[0,1],palette=['#4C78A8','#E45756'],
             bins=60,stat='density',common_norm=False,element='step',fill=False,ax=axs[1])
axs[1].axvline(0,ls='--',color='gray',label='No probability change')
axs[1].set(title='Ensemble − single probability shift by observed class',
           xlabel='Change in estimated fraud probability',ylabel='Within-class density')
leg=axs[1].get_legend()
if leg:leg.set_title('Observed class')
fig.suptitle('Exploratory effect of OOF-based blend selection (not independent test performance)',y=1.01)
finish_plot(fig,'38_blend_vs_single_predictions')

## 10. Generate Kaggle submission files

Use the exact `id` order in `sample_submission.csv`. The `is_fraud` column contains scores between 0 and 1. The primary submission is the OOF-selected strategy. We also export the best individual candidate and a blend over all candidates for comparison. **No Kaggle submission is sent automatically.**

In [ ]:
submissions = {
    'submission.csv': blend_predictions[chosen]['test'],
    'submission_best_single.csv': blend_predictions['best_single']['test'],
    'submission_all_candidates.csv': blend_predictions['all_candidates']['test'],
}
for filename, probabilities in submissions.items():
    file = sample_submission[[ID]].copy()
    file[TARGET] = np.asarray(probabilities, dtype=np.float64).clip(0, 1)
    assert file[ID].equals(sample_submission[ID])
    assert len(file) == len(test) and file[TARGET].between(0, 1).all()
    output_path = OUTPUT_DIR / filename
    file.to_csv(output_path, index=False)
    print('Wrote', output_path, 'rows:', len(file))

display(pd.read_csv(OUTPUT_DIR / 'submission.csv').head())
# Reference manifest; no raw training observations are stored.
manifest = {
    'competition':'playground-series-s6e11',
    'target':TARGET, 'primary_metric':'roc_auc', 'n_splits':N_SPLITS,
    'random_state':RANDOM_STATE, 'fast_dev_run':FAST_DEV_RUN,
    'training_rows':len(X), 'test_rows':len(X_test),
    'chosen_strategy':chosen, 'candidate_count':len(specs),
    'sklearn':sklearn_version, 'xgboost':xgboost.__version__,
    'model_persistence':SAVE_FOLD_MODELS,
}
(OUTPUT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
print('All outputs saved in:', OUTPUT_DIR.resolve())

### 10.1 Submission score sanity check (unlabeled test set)

The distribution of competition-test predictions can be compared with OOF training scores for gross shifts or invalid values, **without knowing test labels**. Matching distributions do not guarantee model generalization. This section does not inspect Kaggle leaderboard feedback.

In [ ]:
test_scores=np.asarray(blend_predictions[chosen]['test'],dtype=float)
train_scores=np.asarray(blend_predictions[chosen]['oof'],dtype=float)
assert np.isfinite(test_scores).all() and np.isfinite(train_scores).all()
assert np.all((test_scores>=0)&(test_scores<=1))
fig,axs=plt.subplots(1,2,figsize=(15,5))
plot_rng=np.random.default_rng(RANDOM_STATE)
tr_scores=train_scores[plot_rng.choice(len(train_scores),size=min(25000,len(train_scores)),replace=False)]
te_scores=test_scores[plot_rng.choice(len(test_scores),size=min(25000,len(test_scores)),replace=False)]
sns.ecdfplot(x=tr_scores,ax=axs[0],label='Train OOF',color='#4C78A8')
sns.ecdfplot(x=te_scores,ax=axs[0],label='Unlabeled test',color='#E45756')
axs[0].set(title='Predicted-risk distribution: training OOF vs Kaggle test',
           xlabel='Predicted fraud probability',ylabel='Cumulative share')
axs[0].legend(title='Prediction source')
summary=[]
for label,values in [('Train OOF',train_scores),('Test',test_scores)]:
    for percentile in [50,90,95,99,99.9]:
        summary.append({'dataset':label,'percentile':percentile,'predicted_probability':float(np.percentile(values,percentile))})
summary_df=pd.DataFrame(summary)
sns.lineplot(data=summary_df,x='percentile',y='predicted_probability',hue='dataset',marker='o',ax=axs[1])
axs[1].set(title='Predicted-risk tail quantiles (unlabeled test included)',
           xlabel='Percentile of model-predicted risk',ylabel='Predicted fraud probability')
axs[1].legend(title='Prediction source')
fig.suptitle('Final submission diagnostic | distributions only, no test outcomes',y=1.01)
finish_plot(fig,'39_submission_probability_sanity')
summary_df.to_csv(OUTPUT_DIR/'submission_probability_quantiles.csv',index=False)

### 10.2 Visualization catalog

Every figure is written to `outputs/figures/`. The catalog below helps locate plots after a long run. Figures are **recomputed on this run**; previous smoke-test plots are not used as substitutes for full-data model results.

In [ ]:
figure_catalog=pd.DataFrame([
    {'figure':f.name,'file_kib':round(f.stat().st_size/1024,1),
     'source_section':('EDA / feature diagnostics' if int(f.name[:2]) in list(range(1,12))+list(range(20,26))
                       else 'OOF model / ensemble / submission diagnostics')}
    for f in sorted(FIGURE_DIR.glob('*.png')) if f.name[:2].isdigit()
])
figure_catalog.to_csv(OUTPUT_DIR/'figure_catalog.csv',index=False)
print(f'Figures currently in {FIGURE_DIR}: {len(figure_catalog)}')
display(figure_catalog)

## 11. Interpretation and limitations

1. **Class imbalance.** With only about 1.67% fraud, raw accuracy and a 0.5 decision threshold can be misleading. Kaggle uses ROC AUC. Report average precision and top-1% capture alongside it.
2. **PCA.** The SVD retention threshold is unsupervised. Low-variance directions can carry important fraud information; PCA is compared against the raw-feature logistic baseline instead of assumed beneficial.
3. **No leakage.** `id` is excluded, and all imputers, encoders, standardizers, and PCA are fitted within each training fold. Keep validation labels away from model-fitting.
4. **Hyperparameter and blend selection.** Selection on CV results makes the reported winning OOF score optimistic. A separate untouched validation set or nested CV would permit less biased performance reporting.
5. **Model averaging versus bagging.** The 5-fold mean is a cross-validation ensemble; the cross-family mean is a probability blend. Random forests themselves combine many randomized trees.
6. **Calibration.** Probability blending can be suboptimal when one model is poorly calibrated (especially when using `class_weight='balanced'`). ROC AUC rewards ranking, not calibration. Brier and log loss help identify problems.
7. **Data governance.** The repository is public. Kaggle's competition rules require publicly shared competition code to also be shared in the competition's Kaggle discussion/notebooks; the rules also restrict sharing competition datasets. Do not commit `data/`, `outputs/`, or publicly publish the notebook before meeting that condition.
8. **Deadline.** Final Kaggle submissions are due November 30, 2026 at 23:59 UTC, according to the competition timeline. Upload `outputs/submission.csv` manually.

**Possible subsequent experiment:** introduce a strictly held-out evaluation split or nested CV and test probability calibration. Preserve the original CV results and notebook instead of overwriting them.

9. **Nonlinearity:** Binned target rates, temporal profiles, and pairwise heatmaps provide descriptive evidence of possible nonlinearities and interactions, not confirmed causal or conditional relationships. Compare standard logistic, spline logistic, random forest, and XGBoost using the same folds.
10. **Outliers:** Conservatively flagged extreme values and Isolation Forest anomalies are exploratory descriptors; they are **not removed** from train/test and should not be equated with fraud. The detector's contamination parameter is not fraud prevalence.
11. **Plots and evidence:** Every visualization is saved in `outputs/figures/`. Numerical group summaries are saved under `outputs/`. Figures made with fixed-seed point samples are explicitly labeled. Avoid reporting notebook EDA as a separately validated experiment or claiming a 95% interval establishes causality.


12. **Assumption audit:** Section 5.1 explicitly distinguishes testable, questionable, and unverifiable logistic assumptions, and Section 8.3 examines OOF calibration, nonlinear specification, and numerical convergence. Avoid saying “all assumptions hold.” No Gaussian predictors or constant Bernoulli variance are required. To avoid redundant dummy columns, logistic pipelines now drop the first category; **rerun all CV fits** rather than comparing previously saved smoke scores to the changed specification.

13. **Offset boosting:** The new `logistic_offset_xgb` family fits one logistic regression and then an XGBoost correction **inside every CV training fold**. Always provide the matching logistic logits as `base_margin` at both fit and inference, including Kaggle test inference. The corrected function is no longer globally linear in the log-odds; the logistic component remains individually interpretable. OOF residual improvements alone do not prove causal nonlinearity, calibration, or better ROC AUC.


14. **Visualization expansion:** Figures 20–39 are exploratory visual audits. Single-variable AUC and class-balanced ECDF/pairplot samples cannot establish incremental contribution or true population prevalence; fixed 1% review budgets are illustrative rather than economic optimization; subgroup metrics are descriptive and require adequate positive/negative counts; model importance is not causal; partial dependence can evaluate unrealistic feature combinations; score differences on Kaggle test data are unlabeled shift checks.
15. **Runtime:** Figures 20–25 run before training; Figures 26–39 run after training and rely on OOF predictions. The partial-dependence figure uses saved fold-1 estimators and a bounded sample; set `RUN_PDP=False` to skip it if resource-constrained. The expanded plots do not change the 15 candidate configurations, five folds, or the chosen submission logic.

16. **Hyperparameter audit:** the latest plots compare candidate-level hyperparameter settings and shared held-out fold scores. Because not every candidate differs by a single hyperparameter, apparent patterns do not identify isolated causal contributions of tuning choices.
